# Week 6 Student Worksheet: Spatial Prediction Shootout

## Traditional Stats vs. Machine Learning: Filling the Gaps

> *"Kriging tells you where it's uncertain. ML just says 'trust me.'"*

### Today's Mission

Transform Week 5's discrete rainfall stations into **continuous rainfall surfaces** using **two fundamentally different approaches**:

1. **Kriging (Statistical)** — uses spatial correlation + provides uncertainty
2. **Random Forest (ML)** — uses data patterns + easily adds features

Then compare them head-to-head with two simpler methods (Nearest Neighbor, IDW) and determine which gives the Commander more actionable intelligence.

> Fill in the code cells marked with `# YOUR CODE HERE`. Use AI tools strategically — but understand every line you write.

In [ ]:
# Install required packages (run once)
%pip install pykrige scikit-learn rasterio rasterstats --quiet

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from shapely.geometry import Point
warnings.filterwarnings('ignore')

# 從 Week 5 複製的函數
def safe_parse_float(value):
    """安全解析浮點數"""
    try:
        if value is None or value == '':
            return 0.0
        return float(value)
    except (ValueError, TypeError):
        return 0.0

def normalize_cwa_simulation_json(data):
    """標準化 CWA 模擬資料格式"""
    normalized = []
    
    try:
        records = data.get('records', {})
        stations = records.get('Station', [])
        
        for station in stations:
            station_name = station.get('StationName', '')
            
            # 提取座標資訊
            geo_info = station.get('GeoInfo', {})
            coordinates = geo_info.get('Coordinates', [])
            
            lat = 0.0
            lon = 0.0
            
            if coordinates and len(coordinates) > 0:
                coord = coordinates[0]
                lat = safe_parse_float(coord.get('StationLatitude'))
                lon = safe_parse_float(coord.get('StationLongitude'))
            
            # 提取行政區資訊
            town_name = geo_info.get('TownName', '')
            county_name = geo_info.get('CountyName', '')
            
            # 提取雨量資訊
            rainfall_element = station.get('RainfallElement', {})
            
            rain_1hr = 0.0
            rain_3hr = 0.0
            rain_24hr = 0.0
            
            if 'Past1hr' in rainfall_element:
                rain_1hr = safe_parse_float(rainfall_element['Past1hr'].get('Precipitation'))
            
            if 'Past3hr' in rainfall_element:
                rain_3hr = safe_parse_float(rainfall_element['Past3hr'].get('Precipitation'))
            
            if 'Past24hr' in rainfall_element:
                rain_24hr = safe_parse_float(rainfall_element['Past24hr'].get('Precipitation'))
            
            # 建立標準化資料
            station_data = {
                'station_name': station_name,
                'county_name': county_name,
                'town_name': f"{county_name}{town_name}",
                'latitude': lat,
                'longitude': lon,
                'rain_1hr': rain_1hr,
                'rain_3hr': rain_3hr,
                'rain_24hr': rain_24hr
            }
            
            normalized.append(station_data)
            
    except Exception as e:
        print(f"✗ CWA 模擬資料標準化失敗: {e}")
    
    return normalized

def parse_rainfall_json(json_file_path):
    """解析雨量 JSON 資料為 GeoDataFrame"""
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # 標準化資料
        normalized_data = normalize_cwa_simulation_json(data)
        
        if not normalized_data:
            print("❌ 無有效資料")
            return None
        
        # 轉換為 DataFrame
        df = pd.DataFrame(normalized_data)
        
        # 過濾有效座標
        valid_coords = (df['latitude'] != 0) & (df['longitude'] != 0)
        df_valid = df[valid_coords].copy()
        
        if len(df_valid) == 0:
            print("❌ 無有效座標資料")
            return None
        
        # 建立幾何點
        geometry = [Point(lon, lat) for lat, lon in zip(df_valid['latitude'], df_valid['longitude'])]
        
        # 建立 GeoDataFrame
        gdf = gpd.GeoDataFrame(
            df_valid,
            geometry=geometry,
            crs='EPSG:4326'  # WGS84
        )
        
        return gdf
        
    except Exception as e:
        print(f"✗ 資料解析失敗: {e}")
        return None

# 載入資料
print("🔄 載入鳳凰颱風資料...")
gdf_all = parse_rainfall_json('fungwong_202511.json')

if gdf_all is not None:
    print(f"✅ 載入 {len(gdf_all)} 個測站資料")
    
    # 篩選花蓮縣 + 宜蘭縣
    target_counties = ['花蓮縣', '宜蘭縣']
    study_rain = gdf_all[gdf_all['county_name'].isin(target_counties)].copy()
    print(f"📍 篩選 {len(study_rain)} 個目標縣市測站")
    
    # 移除無效雨量站
    study_rain = study_rain[study_rain['rain_1hr'] > 0].copy()
    print(f"🌧️ 移除無效雨量站後剩餘 {len(study_rain)} 個測站")
    
    # 轉換至 EPSG:3826
    study_rain_3826 = study_rain.to_crs('EPSG:3826')
    print(f"🗺️ 轉換至 EPSG:3826: {study_rain_3826.crs}")
    
    # 提取座標陣列
    x = study_rain_3826.geometry.x.values  # Easting (meters)
    y = study_rain_3826.geometry.y.values  # Northing (meters)
    z = study_rain_3826['rain_1hr'].values
    
    print(f"\n📊 研究區域測站統計 (rain > 0): {len(study_rain_3826)}")
    print(f"CRS: {study_rain_3826.crs}")
    print(f"\n🏆 前 5 個測站 (時雨量):")
    top5 = study_rain_3826.nlargest(5, 'rain_1hr')[['station_name', 'county_name', 'rain_1hr']]
    print(top5.to_string(index=False))
    
    # 顯示基本統計
    print(f"\n📈 雨量統計:")
    print(f"  平均時雨量: {z.mean():.2f} mm/hr")
    print(f"  最大時雨量: {z.max():.2f} mm/hr")
    print(f"  最小時雨量: {z.min():.2f} mm/hr")
    
else:
    print("❌ 資料載入失敗")
    x, y, z = None, None, None

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from shapely.geometry import Point
warnings.filterwarnings('ignore')

# YOUR CODE HERE:
# 1. Define normalize_cwa_json() and parse_rainfall_json() (copy from Week 5)
# 2. Load 'data/scenarios/fungwong_202511.json'
# 3. Parse into GeoDataFrame
# 4. Filter to 花蓮縣 + 宜蘭縣
# 5. Remove stations with rain_1hr <= 0
# 6. Convert to EPSG:3826


# Extract coordinate arrays for Kriging / ML
# x = study_rain_3826.geometry.x.values  # Easting (meters)
# y = study_rain_3826.geometry.y.values  # Northing (meters)
# z = study_rain_3826['rain_1hr'].values

# print(f"Study area stations (rain > 0): {len(study_rain_3826)}")
# print(f"CRS: {study_rain_3826.crs}")
# print(f"\nTop 5 stations:")
# print(study_rain_3826.nlargest(5, 'rain_1hr')[['station_name', 'county', 'rain_1hr']].to_string(index=False))

from pykrige.ok import OrdinaryKriging

# 🔴 第一次嘗試：對原始降雨資料執行 Kriging
# 這會展示為什麼需要 log transform

if x is not None and y is not None and z is not None:
    print("🔴 第一次嘗試：對原始降雨資料執行 Kriging")
    print("⚠️ 預期會看到不良的 variogram 擬合")
    
    # 計算初始參數
    initial_sill = float(z.var())
    initial_range = 50000.0  # 50km
    initial_nugget = float(z.var() * 0.1)
    
    print(f"📊 初始參數:")
    print(f"  Sill (變異數): {initial_sill:.2f}")
    print(f"  Range (影響範圍): {initial_range/1000:.1f} km")
    print(f"  Nugget (噪音比例): {initial_nugget:.2f}")
    
    try:
        # 建立原始資料的 Kriging 模型
        OK_naive = OrdinaryKriging(x, y, z, variogram_model='spherical',
                                    verbose=False, enable_plotting=True, nlags=15,
                                    variogram_parameters={'sill': initial_sill,
                                                          'range': initial_range,
                                                          'nugget': initial_nugget})
        
        # 取得擬合參數
        params = OK_naive.variogram_model_parameters
        print(f"\n📈 擬合結果:")
        print(f"Sill:   {params[0]:.2f}")
        print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
        print(f"Nugget: {params[2]:.2f}")
        
        print(f"\n⚠️ 觀察 variogram 圖：")
        print(f"  - 紅色點是否遵循黑色曲線？")
        print(f"  - 如果點分散嚴重，表示擬合不良")
        print(f"  - 這通常是由於極端降雨值造成的長尾分佈")
        
        # 計算擬合品質指標
        lags = OK_naive.lags
        semivariance = OK_naive.semivariance
        
        # 簡單的 SSE 計算
        def spherical_model(h, sill, range_, nugget):
            h = np.array(h)
            result = np.zeros_like(h)
            mask = h <= range_
            result[mask] = nugget + sill * (1.5 * (h[mask]/range_) - 0.5 * (h[mask]/range_)**3)
            result[~mask] = nugget + sill
            return result
        
        fitted_values = spherical_model(lags, params[0], params[1], params[2])
        sse = np.sum((semivariance - fitted_values)**2)
        print(f"  - SSE (擬合誤差): {sse:.2f}")
        
    except Exception as e:
        print(f"❌ Kriging 執行失敗: {e}")
        print("這可能表示資料有問題或參數設定不當")
        
else:
    print("❌ 無效的座標或雨量資料，請先執行 Cell [1]")

In [ ]:
from pykrige.ok import OrdinaryKriging

# 🔴 First attempt: run Kriging on raw rainfall data
# YOUR CODE HERE:
# 1. Create OrdinaryKriging with raw z values
#    Use variogram_model='spherical', verbose=False, enable_plotting=True, nlags=15
# 2. Provide initial parameters to help the optimizer:
#    sill = z.var(), range = 50000, nugget = z.var() * 0.1

# initial_sill = float(z.var())
# initial_range = 50000.0
# initial_nugget = float(z.var() * 0.1)

# OK_naive = OrdinaryKriging(x, y, z, variogram_model='spherical',
#                             verbose=False, enable_plotting=True, nlags=15,
#                             variogram_parameters={'sill': initial_sill,
#                                                   'range': initial_range,
#                                                   'nugget': initial_nugget})

# params = OK_naive.variogram_model_parameters
# print(f"Sill:   {params[0]:.1f}")
# print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
# print(f"Nugget: {params[2]:.1f}")
# print("\n⚠️ Look at the plot — do the dots follow the curve?")

# 為什麼會失敗？看看直方圖
# 在責怪工具之前，先看看你的資料

# 建立比較圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：原始降雨值
axes[0].hist(z, bins=30, color='tomato', edgecolor='black', alpha=0.7)
axes[0].set_title('Raw Rainfall (mm/hr)')
axes[0].set_xlabel('Rainfall (mm/hr)')
axes[0].set_ylabel('Station Count')

# 右圖：log(1+z) 轉換後
z_log = np.log1p(z)
axes[1].hist(z_log, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_title('After log(1+z)')
axes[1].set_xlabel('log(1 + Rainfall)')

plt.tight_layout()
plt.show()

print("觀察結果:")
print("左圖：大部分測站 < 10 mm，但少數測站 50-130 mm")
print("右圖：log-transform 後數值更平衡")
print("結論：極端值干擾 variogram，需要 log transform")

In [ ]:
# YOUR CODE HERE:
# 1. Plot a histogram of z (raw rainfall)
# 2. Plot a histogram of np.log1p(z) (log-transformed)
# 3. Compare: which one looks more "balanced"?

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# axes[0].hist(z, bins=30, color='tomato', edgecolor='black')
# axes[0].set_title('Raw Rainfall (mm/hr)')
# axes[0].set_xlabel('Rainfall (mm/hr)')

# z_log = np.log1p(z)
# axes[1].hist(z_log, bins=30, color='steelblue', edgecolor='black')
# axes[1].set_title('After log(1+z)')
# axes[1].set_xlabel('log(1 + Rainfall)')
# plt.tight_layout()
# plt.show()

# print("Left: most stations < 10 mm, but a few are 50-130 mm.")
# print("Those extreme values mess up the variogram.")
# print("Right: after log-transform, the values are more balanced.")

# 🟢 第二次嘗試：對 log-transform 資料執行 Kriging
# 現在應該看到明顯改善的 variogram 擬合

if x is not None and y is not None and z is not None:
    print("🟢 第二次嘗試：對 log-transform 資料執行 Kriging")
    print("✅ 預期會看到明顯改善的 variogram 擬合")
    
    # 使用 log(1+z) 轉換
    z_log = np.log1p(z)
    
    # 計算基於 log-transform 資料的初始參數
    initial_sill = float(z_log.var())
    initial_range = 50000.0  # 50km
    initial_nugget = float(z_log.var() * 0.1)
    
    print(f"📊 Log-transform 後的初始參數:")
    print(f"  Sill (變異數): {initial_sill:.3f}")
    print(f"  Range (影響範圍): {initial_range/1000:.1f} km")
    print(f"  Nugget (噪音比例): {initial_nugget:.3f}")
    
    try:
        # 建立基於 log-transform 資料的 Kriging 模型
        OK = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                            verbose=False, enable_plotting=True, nlags=15,
                            variogram_parameters={'sill': initial_sill,
                                                  'range': initial_range,
                                                  'nugget': initial_nugget})
        
        # 取得擬合參數
        params = OK.variogram_model_parameters
        print(f"\n📈 擬合結果:")
        print(f"Sill:   {params[0]:.3f}")
        print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
        print(f"Nugget: {params[2]:.3f}")
        
        print(f"\n✅ 與 Cell [2a] 比較：")
        print(f"  - 紅色點現在應該更好地遵循黑色曲線")
        print(f"  - SSE (擬合誤差) 應該顯著降低")
        
        # 計算擬合品質指標
        lags = OK.lags
        semivariance = OK.semivariance
        
        # Spherical 模型函數
        def spherical_model(h, sill, range_, nugget):
            h = np.array(h)
            result = np.zeros_like(h)
            mask = h <= range_
            result[mask] = nugget + sill * (1.5 * (h[mask]/range_) - 0.5 * (h[mask]/range_)**3)
            result[~mask] = nugget + sill
            return result
        
        # 計算 SSE
        fitted_values = spherical_model(lags, params[0], params[1], params[2])
        sse = np.sum((semivariance - fitted_values)**2)
        print(f"  - SSE (擬合誤差): {sse:.4f}")
        
        # 計算 R² (擬合優度)
        ss_total = np.sum((semivariance - np.mean(semivariance))**2)
        ss_residual = np.sum((semivariance - fitted_values)**2)
        r_squared = 1 - (ss_residual / ss_total)
        print(f"  - R² (擬合優度): {r_squared:.3f}")
        
        # 解讀參數意義
        print(f"\n🔍 參數解讀:")
        print(f"  Sill ({params[0]:.3f}): log-scale 的最大變異程度")
        print(f"  Range ({params[1]/1000:.1f} km): 空間相關性的影響距離")
        print(f"  Nugget ({params[2]:.3f}): 測量噪音與微小尺度變異")
        
        # 計算實際距離的影響
        print(f"\n📏 實際意義:")
        print(f"  - 在 {params[1]/1000:.1f} km 距離內，測站值具有空間相關性")
        print(f"  - 超過此距離，測站值可視為獨立")
        print(f"  - Nugget/Sill 比例: {params[2]/params[0]*100:.1f}% (測量噪音比例)")
        
        # 儲存模型供後續使用
        global OK_model
        OK_model = OK
        global z_log_global
        z_log_global = z_log
        
        print(f"\n💾 模型已儲存，準備進行網格預測")
        
    except Exception as e:
        print(f"❌ Kriging 執行失敗: {e}")
        print("檢查資料格式或參數設定")
        
else:
    print("❌ 無效的座標或雨量資料，請先執行前面的 Cell")

In [ ]:
# 🟢 Second attempt: Kriging on log-transformed data
# YOUR CODE HERE:
# 1. z_log = np.log1p(z)  (already computed above)
# 2. Create OrdinaryKriging with z_log (not z!)
# 3. Use initial parameters based on z_log.var()

# initial_sill = float(z_log.var())
# initial_range = 50000.0
# initial_nugget = float(z_log.var() * 0.1)

# OK = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
#                       verbose=False, enable_plotting=True, nlags=15,
#                       variogram_parameters={'sill': initial_sill,
#                                             'range': initial_range,
#                                             'nugget': initial_nugget})

# params = OK.variogram_model_parameters
# print(f"Sill:   {params[0]:.3f}")
# print(f"Range:  {params[1]:.0f} m ({params[1]/1000:.1f} km)")
# print(f"Nugget: {params[2]:.3f}")
# print("\n✅ Compare with Cell [2a] — the dots should follow the curve now.")

# 哪個 Variogram 最適合？範圍與模型比較
# 比較不同範圍和模型的效果

if x is not None and y is not None and z is not None:
    print("🔍 Variogram 模型與範圍比較分析")
    
    # 定義測試範圍 (公尺)
    ranges_km = [50, 25, 15]
    ranges_m = [r * 1000 for r in ranges_km]
    models = ['spherical', 'exponential']
    
    # Spherical 模型函數
    def spherical_model(h, sill, range_, nugget):
        h = np.array(h)
        result = np.zeros_like(h)
        mask = h <= range_
        result[mask] = nugget + sill * (1.5 * (h[mask]/range_) - 0.5 * (h[mask]/range_)**3)
        result[~mask] = nugget + sill
        return result
    
    # Exponential 模型函數
    def exponential_model(h, sill, range_, nugget):
        h = np.array(h)
        return nugget + sill * (1 - np.exp(-3 * h / range_))
    
    # 計算 SSE 的函數
    def calculate_sse(lags, semivariance, model_func, params):
        fitted_values = model_func(lags, params[0], params[1], params[2])
        return np.sum((semivariance - fitted_values)**2)
    
    # 儲存結果
    results = {}
    
    # ─── 圖 1: Spherical 模型 × 3 種範圍 ───
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Spherical Model - Range Comparison', fontsize=14, fontweight='bold')
    
    for i, (ax, r_m, r_km) in enumerate(zip(axes, ranges_m, ranges_km)):
        # 建立 Kriging 模型
        ok_test = OrdinaryKriging(x, y, z_log_global, variogram_model='spherical',
                                verbose=False, enable_plotting=False, nlags=15,
                                variogram_parameters={'sill': float(z_log_global.var()),
                                                      'range': r_m,
                                                      'nugget': float(z_log_global.var() * 0.1)})
        
        # 繪製經驗 variogram
        ax.scatter(ok_test.lags/1000, ok_test.semivariance, c='red', s=30, alpha=0.7, zorder=5)
        
        # 繪製擬合曲線
        params = ok_test.variogram_model_parameters
        h_range = np.linspace(0, max(ok_test.lags)/1000, 100)
        fitted_curve = spherical_model(h_range * 1000, params[0], params[1], params[2])
        ax.plot(h_range, fitted_curve, 'k-', linewidth=2)
        
        # 計算 SSE
        sse = calculate_sse(ok_test.lags, ok_test.semivariance, spherical_model, params)
        
        # 設定圖表
        ax.set_title(f'Range = {r_km} km', fontsize=12, fontweight='bold')
        ax.set_xlabel('Distance (km)')
        ax.set_ylabel('Semivariance')
        ax.grid(True, alpha=0.3)
        ax.text(0.05, 0.95, f'SSE: {sse:.3f}', transform=ax.transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), va='top')
        
        # 儲存結果
        results[f'spherical_{r_km}km'] = {
            'model': 'spherical',
            'range_km': r_km,
            'params': params,
            'sse': sse
        }
    
    plt.tight_layout()
    plt.show()
    
    # ─── 圖 2: Exponential 模型 × 3 種範圍 ───
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Exponential Model - Range Comparison', fontsize=14, fontweight='bold')
    
    for i, (ax, r_m, r_km) in enumerate(zip(axes, ranges_m, ranges_km)):
        # 建立 Kriging 模型
        ok_test = OrdinaryKriging(x, y, z_log_global, variogram_model='exponential',
                                verbose=False, enable_ploting=False, nlags=15,
                                variogram_parameters={'sill': float(z_log_global.var()),
                                                      'range': r_m,
                                                      'nugget': float(z_log_global.var() * 0.1)})
        
        # 繪製經驗 variogram
        ax.scatter(ok_test.lags/1000, ok_test.semivariance, c='red', s=30, alpha=0.7, zorder=5)
        
        # 繪製擬合曲線
        params = ok_test.variogram_model_parameters
        h_range = np.linspace(0, max(ok_test.lags)/1000, 100)
        fitted_curve = exponential_model(h_range * 1000, params[0], params[1], params[2])
        ax.plot(h_range, fitted_curve, 'k-', linewidth=2)
        
        # 計算 SSE
        sse = calculate_sse(ok_test.lags, ok_test.semivariance, exponential_model, params)
        
        # 設定圖表
        ax.set_title(f'Range = {r_km} km', fontsize=12, fontweight='bold')
        ax.set_xlabel('Distance (km)')
        ax.set_ylabel('Semivariance')
        ax.grid(True, alpha=0.3)
        ax.text(0.05, 0.95, f'SSE: {sse:.3f}', transform=ax.transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), va='top')
        
        # 儲存結果
        results[f'exponential_{r_km}km'] = {
            'model': 'exponential',
            'range_km': r_km,
            'params': params,
            'sse': sse
        }
    
    plt.tight_layout()
    plt.show()
    
    # ─── 總結表格 ───
    print(f"\n📊 擬合結果總結:")
    print(f"{'模型':<12} {'範圍(km)':<8} {'SSE':<8} {'Sill':<8} {'Range(m)':<10} {'Nugget':<8}")
    print(f"{'-'*60}")
    
    best_overall = None
    best_sse = float('inf')
    
    for key, result in results.items():
        model = result['model']
        range_km = result['range_km']
        sse = result['sse']
        params = result['params']
        
        print(f"{model:<12} {range_km:<8} {sse:<8.3f} {params[0]:<8.3f} {params[1]:<10.0f} {params[2]:<8.3f}")
        
        if sse < best_sse:
            best_sse = sse
            best_overall = key
    
    print(f"{'-'*60}")
    print(f"🏆 最佳擬合: {best_overall} (SSE = {best_sse:.3f})")
    
    # ─── 分析與洞察 ───
    print(f"\n🔍 分析洞察:")
    
    # Spherical 模型內部比較
    spherical_results = [r for k, r in results.items() if r['model'] == 'spherical']
    best_spherical = min(spherical_results, key=lambda x: x['sse'])
    print(f"1. Spherical 模型內部:")
    print(f"   最佳範圍: {best_spherical['range_km']} km (SSE: {best_spherical['sse']:.3f})")
    
    # Exponential 模型內部比較
    exp_results = [r for k, r in results.items() if r['model'] == 'exponential']
    best_exp = min(exp_results, key=lambda x: x['sse'])
    print(f"2. Exponential 模型內部:")
    print(f"   最佳範圍: {best_exp['range_km']} km (SSE: {best_exp['sse']:.3f})")
    
    # 模型間比較
    print(f"3. 模型間比較 (相同範圍):")
    for r_km in ranges_km:
        sph_key = f'spherical_{r_km}km'
        exp_key = f'exponential_{r_km}km'
        if sph_key in results and exp_key in results:
            sph_sse = results[sph_key]['sse']
            exp_sse = results[exp_key]['sse']
            winner = 'Spherical' if sph_sse < exp_sse else 'Exponential'
            diff = abs(sph_sse - exp_sse)
            print(f"   {r_km}km: {winner} 優勝 (差異: {diff:.3f})")
    
    # 儲存最佳模型
    global best_model
    best_model = results[best_overall]
    print(f"\n💾 最佳模型已儲存供後續使用")
    
else:
    print("❌ 無效資料，請先執行前面的 Cell")

In [ ]:
# YOUR CODE HERE:
# 1. Define ranges: [50000, 25000, 15000]
# 2. Figure 1: Spherical × 3 Ranges (1×3 subplot)
#    For each: create OrdinaryKriging, plot lags vs semivariance (red dots),
#    plot fitted curve (black line), compute SSE
# 3. Figure 2: Exponential × 3 Ranges (1×3 subplot)
#    Same as above but with variogram_model='exponential'
# 4. Print summary table:
#    - Compare within Spherical (Range effect)
#    - Compare within Exponential (Range effect)
#    - Compare Spherical vs Exponential at same Range (Model effect)

# ranges_km = [50, 25, 15]

# ─── Figure 1: Spherical ───
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# for ax, rkm in zip(axes, ranges_km):
#     ok_test = OrdinaryKriging(x, y, z_log, variogram_model='spherical', ...)
#     ax.scatter(ok_test.lags/1000, ok_test.semivariance, c='red', ...)
#     # plot fitted curve ...

# ─── Figure 2: Exponential ───
# (same structure)

# 💡 Questions:
#   1. Within Spherical, which Range gives the best fit?
#   2. Within Exponential, which Range gives the best fit?
#   3. At the same Range, does model choice matter much?

import time

# 定義插值網格並執行 Kriging
# 建立覆蓋研究區域的規則網格，解析度 1000m

if x is not None and y is not None and z is not None:
    print("🗺️ 定義插值網格並執行 Kriging 預測")
    
    # 網格參數設定
    buffer_m = 5000  # 5km 緩衝區
    resolution = 1000  # 1000m 解析度
    
    # 計算網格範圍（添加緩衝區）
    x_min = x.min() - buffer_m
    x_max = x.max() + buffer_m
    y_min = y.min() - buffer_m
    y_max = y.max() + buffer_m
    
    # 建立規則網格
    grid_x = np.arange(x_min, x_max, resolution)
    grid_y = np.arange(y_min, y_max, resolution)
    
    print(f"📏 網格設定:")
    print(f"  X 範圍: {x_min/1000:.1f} - {x_max/1000:.1f} km")
    print(f"  Y 範圍: {y_min/1000:.1f} - {y_max/1000:.1f} km")
    print(f"  解析度: {resolution} m")
    print(f"  網格大小: {len(grid_x)}×{len(grid_y)} = {len(grid_x)*len(grid_y):,} 點")
    
    # 使用最佳模型執行 Kriging
    if 'best_model' in globals():
        model_type = best_model['model']
        range_m = best_model['params'][1]
        sill = best_model['params'][0]
        nugget = best_model['params'][2]
        
        print(f"\n🎯 使用最佳模型: {model_type}")
        print(f"  Range: {range_m/1000:.1f} km")
        print(f"  Sill: {sill:.3f}")
        print(f"  Nugget: {nugget:.3f}")
        
        # 建立最佳 Kriging 模型
        OK_final = OrdinaryKriging(x, y, z_log_global, variogram_model=model_type,
                                  verbose=False, enable_plotting=False, nlags=15,
                                  variogram_parameters={'sill': sill, 'range': range_m, 'nugget': nugget})
    else:
        # 使用預設模型
        print(f"\n⚠️ 使用預設 Spherical 模型 (50km)")
        OK_final = OK_model
    
    # 執行 Kriging 預測
    print(f"\n⏱️ 執行 Kriging 預測...")
    t0 = time.time()
    
    try:
        # 在 log-space 執行預測
        z_kriging_log, ss_kriging_log = OK_final.execute('grid', grid_x, grid_y)
        
        prediction_time = time.time() - t0
        print(f"✅ Kriging (log-space) 完成，耗時 {prediction_time:.1f} 秒")
        
        # Back-transform 到真實降雨值 (mm/hr)
        print(f"🔄 Back-transform 到真實降雨單位...")
        z_kriging = np.expm1(z_kriging_log)
        
        # 確保降雨值不為負
        z_kriging[z_kriging < 0] = 0
        
        # 保留 log-space 的方差用於 Sigma Map
        ss_kriging = ss_kriging_log
        
        print(f"📊 預測結果統計:")
        print(f"  降雨範圍: {np.nanmin(z_kriging):.2f} - {np.nanmax(z_kriging):.2f} mm/hr")
        print(f"  平均降雨: {np.nanmean(z_kriging):.2f} mm/hr")
        print(f"  方差範圍: {np.nanmin(ss_kriging):.4f} - {np.nanmax(ss_kriging):.4f}")
        print(f"  平均方差: {np.nanmean(ss_kriging):.4f}")
        
        # 檢查預測合理性
        original_max = z.max()
        predicted_max = np.nanmax(z_kriging)
        print(f"\n🔍 合理性檢查:")
        print(f"  原始最大降雨: {original_max:.1f} mm/hr")
        print(f"  預測最大降雨: {predicted_max:.1f} mm/hr")
        print(f"  差異: {predicted_max - original_max:+.1f} mm/hr")
        
        if predicted_max > original_max * 1.5:
            print(f"  ⚠️ 預測值可能過高，檢查模型參數")
        elif predicted_max < original_max * 0.5:
            print(f"  ⚠️ 預測值可能過低，檢查模型參數")
        else:
            print(f"  ✅ 預測值在合理範圍內")
        
        # 儲存網格資訊供後續使用
        global grid_info
        grid_info = {
            'x_min': x_min, 'x_max': x_max,
            'y_min': y_min, 'y_max': y_max,
            'grid_x': grid_x, 'grid_y': grid_y,
            'resolution': resolution,
            'shape': z_kriging.shape
        }
        
        # 儲存預測結果
        global kriging_results
        kriging_results = {
            'z_kriging': z_kriging,
            'ss_kriging': ss_kriging,
            'z_kriging_log': z_kriging_log,
            'ss_kriging_log': ss_kriging_log
        }
        
        print(f"\n💾 網格資訊和預測結果已儲存")
        
        # 視覺化快速檢查
        plt.figure(figsize=(10, 8))
        plt.imshow(z_kriging, extent=[x_min/1000, x_max/1000, y_min/1000, y_max/1000],
                  origin='lower', cmap='YlOrRd', vmin=0, vmax=max(z)*1.1)
        plt.colorbar(label='Rainfall (mm/hr)')
        plt.scatter(x/1000, y/1000, c='black', s=20, zorder=5, label='Stations')
        plt.xlabel('Eastings (km)')
        plt.ylabel('Northings (km)')
        plt.title('Kriging Prediction - Quick Check')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
        
    except Exception as e:
        print(f"❌ Kriging 預測失敗: {e}")
        print("檢查網格設定或模型參數")
        
else:
    print("❌ 無效資料，請先執行前面的 Cell")

In [ ]:
import time

# YOUR CODE HERE:
# 1. Calculate grid extent from x, y arrays (with 5km buffer)
# 2. Create grid_x and grid_y using np.arange with 1000m step
# 3. Execute Kriging in log-space: z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
# 4. Back-transform: z_kriging = np.expm1(z_kriging_log)

buffer_m = 5000
resolution = 1000  # meters — use 500 for finer resolution (slower)

# x_min = x.min() - buffer_m
# x_max = x.max() + buffer_m
# y_min = y.min() - buffer_m
# y_max = y.max() + buffer_m
# grid_x = np.arange(x_min, x_max, resolution)
# grid_y = np.arange(y_min, y_max, resolution)

# print(f"Grid: {len(grid_x)}×{len(grid_y)} = {len(grid_x)*len(grid_y):,} points @ {resolution}m")

# t0 = time.time()
# z_kriging_log, ss_kriging_log = OK.execute('grid', grid_x, grid_y)
# print(f"✓ Kriging (log-space) done in {time.time()-t0:.1f}s")

# # Back-transform to real rainfall (mm/hr)
# z_kriging = np.expm1(z_kriging_log)
# z_kriging[z_kriging < 0] = 0
# ss_kriging = ss_kriging_log  # keep log-space variance for Sigma Map

# print(f"  z range (mm/hr): {np.nanmin(z_kriging):.1f} - {np.nanmax(z_kriging):.1f}")

from sklearn.ensemble import RandomForestRegressor

# 機器學習 - Random Forest 預測
# Captain's Log: 將坐標視為輸入特徵，ML 學習 f(easting, northing) → rainfall

if x is not None and y is not None and z is not None:
    print("🤖 機器學習 - Random Forest 預測")
    print("🎯 將坐標視為輸入特徵，學習空間降雨模式")
    
    # 準備訓練資料
    X_train = np.column_stack([x, y])  # [easting, northing]
    y_train = z
    
    print(f"📊 訓練資料:")
    print(f"  特徵矩陣 X_train: {X_train.shape}")
    print(f"  目標向量 y_train: {y_train.shape}")
    print(f"  特徵: [Easting (m), Northing (m)]")
    
    # 建立並訓練 Random Forest 模型
    print(f"\n🌳 建立 Random Forest 模型...")
    rf = RandomForestRegressor(
        n_estimators=200,        # 樹的數量
        min_samples_leaf=3,      # 葉節點最小樣本數
        random_state=42,         # 隨機種子
        n_jobs=-1               # 使用所有 CPU 核心
    )
    
    # 訓練模型
    t0 = time.time()
    rf.fit(X_train, y_train)
    training_time = time.time() - t0
    
    print(f"✅ 模型訓練完成，耗時 {training_time:.2f} 秒")
    print(f"📈 訓練 R²: {rf.score(X_train, y_train):.3f}")
    
    # 在相同網格上預測
    if 'grid_info' in globals():
        grid_x = grid_info['grid_x']
        grid_y = grid_info['grid_y']
        
        print(f"\n🗺️ 在網格上預測...")
        print(f"  網格大小: {len(grid_x)}×{len(grid_y)}")
        
        # 建立網格坐標
        grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)
        X_grid = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])
        
        print(f"  預測點數: {X_grid.shape[0]:,}")
        
        # 執行預測
        t0 = time.time()
        z_rf = rf.predict(X_grid).reshape(grid_xx.shape)
        prediction_time = time.time() - t0
        
        print(f"✅ Random Forest 預測完成，耗時 {prediction_time:.2f} 秒")
        
        # 預測結果統計
        print(f"📊 預測結果統計:")
        print(f"  降雨範圍: {z_rf.min():.2f} - {z_rf.max():.2f} mm/hr")
        print(f"  平均降雨: {z_rf.mean():.2f} mm/hr")
        print(f"  預測為零的點: {np.sum(z_rf == 0):,} ({np.sum(z_rf == 0)/z_rf.size*100:.1f}%)")
        
        # 與原始資料比較
        print(f"\n🔍 與原始資料比較:")
        print(f"  原始範圍: {z.min():.2f} - {z.max():.2f} mm/hr")
        print(f"  預測範圍: {z_rf.min():.2f} - {z_rf.max():.2f} mm/hr")
        
        # 檢查極端值處理
        original_extreme = z.max()
        predicted_extreme = z_rf.max()
        print(f"  極端值保留: {predicted_extreme/original_extreme*100:.1f}%")
        
        # 儲存結果
        global rf_results
        rf_results = {
            'z_rf': z_rf,
            'model': rf,
            'training_time': training_time,
            'prediction_time': prediction_time
        }
        
        print(f"\n💾 Random Forest 結果已儲存")
        
        # 視覺化快速檢查
        plt.figure(figsize=(10, 8))
        plt.imshow(z_rf, extent=[grid_info['x_min']/1000, grid_info['x_max']/1000, 
                                grid_info['y_min']/1000, grid_info['y_max']/1000],
                  origin='lower', cmap='YlOrRd', vmin=0, vmax=max(z)*1.1)
        plt.colorbar(label='Rainfall (mm/hr)')
        plt.scatter(x/1000, y/1000, c='black', s=20, zorder=5, label='Stations')
        plt.xlabel('Eastings (km)')
        plt.ylabel('Northings (km)')
        plt.title('Random Forest Prediction - Quick Check')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
        
        # 訓練資料上的預測 vs 實際值比較
        y_train_pred = rf.predict(X_train)
        
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.scatter(y_train, y_train_pred, alpha=0.6, s=30)
        plt.plot([0, max(y_train)], [0, max(y_train)], 'r--', linewidth=2)
        plt.xlabel('Actual Rainfall (mm/hr)')
        plt.ylabel('Predicted Rainfall (mm/hr)')
        plt.title('Training Data: Actual vs Predicted')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        residuals = y_train - y_train_pred
        plt.scatter(y_train_pred, residuals, alpha=0.6, s=30)
        plt.axhline(y=0, color='r', linestyle='--')
        plt.xlabel('Predicted Rainfall (mm/hr)')
        plt.ylabel('Residuals (mm/hr)')
        plt.title('Residuals Analysis')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 訓練品質指標
        from sklearn.metrics import mean_squared_error, r2_score
        mse = mean_squared_error(y_train, y_train_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_train, y_train_pred)
        
        print(f"\n📈 訓練品質指標:")
        print(f"  MSE: {mse:.2f}")
        print(f"  RMSE: {rmse:.2f} mm/hr")
        print(f"  R²: {r2:.3f}")
        
    else:
        print("❌ 網格資訊不存在，請先執行 Kriging Cell")
        
else:
    print("❌ 無效資料，請先執行前面的 Cell")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# YOUR CODE HERE:
# 1. Prepare features: X_train = np.column_stack([x, y])
# 2. Train RandomForestRegressor
# 3. Create meshgrid from grid_x, grid_y
# 4. Predict on the grid

# X_train = np.column_stack([x, y])
# y_train = z

# rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
# rf.fit(X_train, y_train)
# print(f"RF Training R²: {rf.score(X_train, y_train):.3f}")

# grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)
# X_grid = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])

# t0 = time.time()
# z_rf = rf.predict(X_grid).reshape(grid_xx.shape)
# print(f"✓ Random Forest done in {time.time()-t0:.1f}s")
# print(f"  z range: {z_rf.min():.1f} - {z_rf.max():.1f} mm/hr")

# ML 黑盒打開 - 特徵重要性分析
# Captain's Question: "AI 使用什麼來預測洪水 - 緯度還是高度？"

if 'rf_results' in globals():
    print("🔍 ML 特徵重要性分析")
    print("🎯 即使 ML 是「黑盒」，我們也可以透過 feature_importances_ 窺視內部")
    
    rf = rf_results['model']
    
    # 取得特徵重要性
    importances = rf.feature_importances_
    
    print(f"📊 特徵重要性:")
    print(f"  Easting (X):  {importances[0]:.3f} ({importances[0]*100:.1f}%)")
    print(f"  Northing (Y): {importances[1]:.3f} ({importances[1]*100:.1f}%)")
    
    # 判斷哪個維度更重要
    dominant_feature = 'easting' if importances[0] > importances[1] else 'northing'
    importance_ratio = max(importances) / min(importances)
    
    print(f"\n🏆 主導特徵: {dominant_feature}")
    print(f"📏 重要性比例: {importance_ratio:.1f}:1")
    
    # 視覺化特徵重要性
    plt.figure(figsize=(8, 6))
    features = ['Easting (X)\n東西向座標', 'Northing (Y)\n南北向座標']
    colors = ['#FF6B6B', '#4ECDC4']
    
    bars = plt.bar(features, importances, color=colors, alpha=0.8, edgecolor='black')
    plt.ylabel('特徵重要性', fontsize=12)
    plt.title('Random Forest 特徵重要性\n颱風 Fung-wong 降雨預測', fontsize=14, fontweight='bold')
    plt.ylim(0, max(importances) * 1.2)
    
    # 添加數值標籤
    for bar, imp in zip(bars, importances):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{imp:.3f}\n({imp*100:.1f}%)', 
                ha='center', va='bottom', fontweight='bold')
    
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    # 物理解釋分析
    print(f"\n🌪️ 物理解釋:")
    print(f"颱風 Fung-wong 的降雨模式分析:")
    
    if dominant_feature == 'easting':
        print(f"  📍 東西向 (Easting) 更重要 → 降雨主要呈東西向分佈")
        print(f"  💡 這符合颱風風暴的典型特徵:")
        print(f"     - 颱風眼牆影響呈東西向帶狀分佈")
        print(f"     - 地形抬升作用在東西向更顯著")
        print(f"     - 花蓮-宜蘭地區的南北向山脈阻擋效應")
    else:
        print(f"  📍 南北向 (Northing) 更重要 → 降雨主要呈南北向分佈")
        print(f"  💡 這可能表示:")
        print(f"     - 颱風路徑主要呈南北向移動")
        print(f"     - 地形影響在南北向更顯著")
        print(f"     - 季風環流與颱風交互作用")
    
    # 與實際地理對應
    print(f"\n🗺️ 地理對應:")
    print(f"研究區域: 花蓮縣 + 宜蘭縣")
    print(f"座標系統: EPSG:3826 (TWD97 TM2)")
    
    if dominant_feature == 'easting':
        print(f"東西向範圍: {x.min()/1000:.0f} - {x.max()/1000:.0f} km")
        print(f"南北向範圍: {y.min()/1000:.0f} - {y.max()/1000:.0f} km")
        print(f"🌊 東西向重要性可能反映:")
        print(f"   - 太平洋水汽東西向輸送")
        print(f"   - 中央山脈東西向地形差異")
        print(f"   - 颱風中心東西向移動路徑")
    
    # 決策含義
    print(f"\n⚡ 決策含義:")
    print(f"對防災指揮官的意義:")
    print(f"  🎯 {dominant_feature} 向的降雨梯度更大")
    print(f"  📍 預警系統應優先考慮{dominant_feature}向的監測站")
    print(f"  🚨 疏散路線規劃應考慮{dominant_feature}向的降雨差異")
    
    # 模型不確定性討論
    print(f"\n⚠️ 模型限制:")
    print(f"  🔸 特徵重要性基於訓練資料統計")
    print(f"  🔸 僅使用坐標特徵，未考慮地形、海拔等因素")
    print(f"  🔸 不同颱風路徑可能產生不同的特徵重要性")
    print(f"  🔸 重要性比例 {importance_ratio:.1f}:1 表示兩個特徵都有貢獻")
    
    # 建議改進方向
    print(f"\n💡 模型改進建議:")
    print(f"  🌟 加入地形特徵: 海拔高度、坡度")
    print(f"  🌟 加入氣象特徵: 風速、風向")
    print(f"  🌟 加入時間特徵: 颱風相對位置")
    print(f"  🌟 使用更複雜的模型: Gradient Boosting, Neural Networks")
    
    # 儲存分析結果
    global feature_analysis
    feature_analysis = {
        'dominant_feature': dominant_feature,
        'importance_ratio': importance_ratio,
        'easting_importance': importances[0],
        'northing_importance': importances[1],
        'interpretation': f"颱風降雨主要呈{dominant_feature}向分佈"
    }
    
    print(f"\n💾 特徵重要性分析結果已儲存")
    
else:
    print("❌ Random Forest 模型不存在，請先執行 ML Cell")

In [ ]:
# YOUR CODE HERE:
# 1. Print rf.feature_importances_
# 2. Interpret: which dimension matters more for typhoon rainfall?

# importances = rf.feature_importances_
# print("Feature Importance:")
# print(f"  Easting (X):  {importances[0]:.3f}")
# print(f"  Northing (Y): {importances[1]:.3f}")
# print(f"\nThe model relies mostly on {'easting' if importances[0] > importances[1] else 'northing'}.")
# print("Think: does this make physical sense for Typhoon Fung-wong?")

from scipy.interpolate import NearestNDInterpolator
from scipy.spatial.distance import cdist

# Lab 1: 四種方法內插對決
# Nearest Neighbor + IDW 基準方法

if x is not None and y is not None and z is not None and 'grid_info' in globals():
    print("🎯 Lab 1: 四種方法內插對決 - 基準方法實作")
    print("📐 計算 Nearest Neighbor 和 IDW，與 Kriging 和 RF 比較")
    
    grid_x = grid_info['grid_x']
    grid_y = grid_info['grid_y']
    
    # 建立網格坐標
    grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)
    grid_points = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])
    station_points = np.column_stack([x, y])
    
    print(f"📊 計算設定:")
    print(f"  網格點數: {grid_points.shape[0]:,}")
    print(f"  測站數量: {len(station_points)}")
    
    # ─── Nearest Neighbor 插值 ───
    print(f"\n🔵 Nearest Neighbor 插值...")
    t0 = time.time()
    
    nn_interp = NearestNDInterpolator(list(zip(x, y)), z)
    z_nn = nn_interp(grid_xx, grid_yy)
    
    nn_time = time.time() - t0
    print(f"✅ Nearest Neighbor 完成，耗時 {nn_time:.2f} 秒")
    print(f"  範圍: {z_nn.min():.2f} - {z_nn.max():.2f} mm/hr")
    print(f"  平均: {z_nn.mean():.2f} mm/hr")
    
    # ─── IDW (Inverse Distance Weighting) 插值 ───
    print(f"\n🟡 IDW (Inverse Distance Weighting) 插值...")
    t0 = time.time()
    
    # 計算距離矩陣
    print(f"  計算距離矩陣...")
    distances = cdist(grid_points, station_points)
    
    # 避免除以零
    distances[distances < 1] = 1
    
    # IDW 權重計算 (power = 2)
    power = 2
    weights = 1.0 / (distances ** power)
    
    # 加權平均
    print(f"  計算加權平均...")
    z_idw = (weights @ z) / weights.sum(axis=1)
    z_idw = z_idw.reshape(grid_xx.shape)
    
    idw_time = time.time() - t0
    print(f"✅ IDW 完成，耗時 {idw_time:.2f} 秒")
    print(f"  範圍: {z_idw.min():.2f} - {z_idw.max():.2f} mm/hr")
    print(f"  平均: {z_idw.mean():.2f} mm/hr")
    
    # ─── 方法比較統計 ───
    print(f"\n📈 四種方法統計比較:")
    print(f"{'方法':<15} {'最小值':<8} {'最大值':<8} {'平均值':<8} {'計算時間':<10}")
    print(f"{'-'*55}")
    
    # 收集所有方法結果
    methods_stats = []
    
    if 'kriging_results' in globals():
        z_kriging = kriging_results['z_kriging']
        methods_stats.append({
            'name': 'Kriging',
            'min': z_kriging.min(),
            'max': z_kriging.max(),
            'mean': z_kriging.mean(),
            'time': 'N/A'
        })
    
    if 'rf_results' in globals():
        z_rf = rf_results['z_rf']
        methods_stats.append({
            'name': 'Random Forest',
            'min': z_rf.min(),
            'max': z_rf.max(),
            'mean': z_rf.mean(),
            'time': f"{rf_results['prediction_time']:.1f}s"
        })
    
    methods_stats.extend([
        {
            'name': 'Nearest Neighbor',
            'min': z_nn.min(),
            'max': z_nn.max(),
            'mean': z_nn.mean(),
            'time': f"{nn_time:.1f}s"
        },
        {
            'name': 'IDW',
            'min': z_idw.min(),
            'max': z_idw.max(),
            'mean': z_idw.mean(),
            'time': f"{idw_time:.1f}s"
        }
    ])
    
    # 顯示統計表格
    for stat in methods_stats:
        print(f"{stat['name']:<15} {stat['min']:<8.2f} {stat['max']:<8.2f} {stat['mean']:<8.2f} {stat['time']:<10}")
    
    # ─── 快速視覺化檢查 ───
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('四種內插方法快速檢查', fontsize=16, fontweight='bold')
    
    # 統一色彩範圍
    vmax = max(z.max(), z_nn.max(), z_idw.max()) * 1.1
    extent = [grid_info['x_min']/1000, grid_info['x_max']/1000, 
              grid_info['y_min']/1000, grid_info['y_max']/1000]
    
    # 繪製各種方法
    method_data = []
    if 'kriging_results' in globals():
        method_data.append(('Kriging', kriging_results['z_kriging']))
    if 'rf_results' in globals():
        method_data.append(('Random Forest', rf_results['z_rf']))
    method_data.extend([('Nearest Neighbor', z_nn), ('IDW', z_idw)])
    
    for i, (ax, (title, data)) in enumerate(zip(axes.flatten(), method_data)):
        im = ax.imshow(data, extent=extent, origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
        ax.scatter(x/1000, y/1000, c='black', s=15, zorder=5, alpha=0.7)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Eastings (km)')
        ax.set_ylabel('Northings (km)')
        ax.grid(True, alpha=0.3)
        plt.colorbar(im, ax=ax, shrink=0.8, label='mm/hr')
    
    plt.tight_layout()
    plt.show()
    
    # ─── 方法特徵分析 ───
    print(f"\n🔍 方法特徵分析:")
    
    print(f"Nearest Neighbor:")
    print(f"  🎯 特徵: Voronoi 圖效果，每個網格點採用最近測站值")
    print(f"  📈 優點: 計算快速，保守估計")
    print(f"  ⚠️ 缺點: 不連續的「補丁」模式，不夠平滑")
    
    print(f"\nIDW (Inverse Distance Weighting):")
    print(f"  🎯 特徵: 距離越近權重越大，呈現「牛眼」效應")
    print(f"  📈 優點: 考慮距離影響，比 NN 平滑")
    print(f"  ⚠️ 缺點: power 參數敏感，可能產生極端值")
    
    # 儲存基準方法結果
    global baseline_results
    baseline_results = {
        'z_nn': z_nn,
        'z_idw': z_idw,
        'nn_time': nn_time,
        'idw_time': idw_time
    }
    
    print(f"\n💾 基準方法結果已儲存")
    print(f"🚀 準備進行四種方法的詳細比較")
    
else:
    print("❌ 缺少必要資料，請先執行前面的 Cell")

In [ ]:
from scipy.interpolate import NearestNDInterpolator
from scipy.spatial.distance import cdist

# YOUR CODE HERE:
# 1. Nearest Neighbor interpolation
# 2. IDW interpolation (手動實作，power=2)
#    ⚠️ 注意：不要用 Rbf(function='inverse')，它不是真正的 IDW，會產生極端值

# nn_interp = NearestNDInterpolator(list(zip(x, y)), z)
# z_nn = nn_interp(grid_xx, grid_yy)

# pts = np.column_stack([x, y])
# grid_pts = np.column_stack([grid_xx.ravel(), grid_yy.ravel()])
# dists = cdist(grid_pts, pts)
# dists[dists < 1] = 1  # 避免除以零
# power = 2
# weights = 1.0 / (dists ** power)
# z_idw = ((weights @ z) / weights.sum(axis=1)).reshape(grid_xx.shape)

# print("✓ Nearest Neighbor + IDW computed")

# 四種方法並排比較
# 建立專業的四方法比較圖，儲存為 interpolation_shootout.png

if 'grid_info' in globals():
    print("🎨 四種方法並排比較 - 專業視覺化")
    print("📸 儲存為 interpolation_shootout.png")
    
    # 收集所有方法資料
    methods_data = []
    
    if 'kriging_results' in globals():
        methods_data.append(('Ordinary Kriging\n(Smooth + Sigma Map)', kriging_results['z_kriging']))
    
    if 'rf_results' in globals():
        methods_data.append(('Random Forest\n(ML "Block" Artifacts)', rf_results['z_rf']))
    
    if 'baseline_results' in globals():
        methods_data.append(('Nearest Neighbor\n(Voronoi / "Patchwork")', baseline_results['z_nn']))
        methods_data.append(('IDW\n(Bullseye Effect)', baseline_results['z_idw']))
    
    if len(methods_data) < 4:
        print("❌ 缺少部分方法結果，請確保所有 Cell 都已執行")
        print(f"目前可用方法: {len(methods_data)}")
    else:
        # 建立 2×2 子圖
        fig, axes = plt.subplots(2, 2, figsize=(18, 14))
        fig.suptitle('Typhoon Fung-wong — Four Interpolation Methods Compared', 
                     fontsize=16, fontweight='bold', y=0.98)
        
        # 統一設定
        vmax = max(z.max(), 
                  max([data.max() for _, data in methods_data])) * 1.1
        extent = [grid_info['x_min']/1000, grid_info['x_max']/1000, 
                  grid_info['y_min']/1000, grid_info['y_max']/1000]
        
        # 繪製每種方法
        for ax, (title, data) in zip(axes.flatten(), methods_data):
            # 主要圖像
            im = ax.imshow(data, extent=extent, origin='lower', 
                          cmap='YlOrRd', vmin=0, vmax=vmax)
            
            # 叠加測站位置
            ax.scatter(x/1000, y/1000, c='black', s=8, zorder=5, alpha=0.8)
            
            # 設定標題和標籤
            ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
            ax.set_xlabel('Eastings (km)', fontsize=10)
            ax.set_ylabel('Northings (km)', fontsize=10)
            
            # 添加網格
            ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
            
            # 添加 colorbar
            cbar = plt.colorbar(im, ax=ax, shrink=0.7, label='mm/hr')
            cbar.ax.tick_params(labelsize=8)
            
            # 添加統計資訊
            stats_text = f'Min: {data.min():.1f}\nMax: {data.max():.1f}\nMean: {data.mean():.1f}'
            ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   verticalalignment='top', fontsize=8)
        
        plt.tight_layout()
        
        # 儲存圖片
        output_filename = 'interpolation_shootout.png'
        plt.savefig(output_filename, dpi=150, bbox_inches='tight', facecolor='white')
        print(f"✅ 圖片已儲存: {output_filename}")
        
        plt.show()
        
        # ─── 方法特徵詳細分析 ───
        print(f"\n🔍 四種方法特徵分析:")
        
        method_analysis = {
            'Nearest Neighbor': {
                'visual': 'Voronoi 補丁圖案',
                'smoothness': '不連續',
                'extremes': '完全保留',
                'artifacts': '邊界不連續',
                'best_for': '保守估計，快速預覽'
            },
            'IDW': {
                'visual': '牛眼效應',
                'smoothness': '中等平滑',
                'extremes': '部分稀釋',
                'artifacts': '測站周圍圓形模式',
                'best_for': '平衡速度與品質'
            },
            'Ordinary Kriging': {
                'visual': '平滑自然',
                'smoothness': '高度平滑',
                'extremes': '適度保留',
                'artifacts': '無明顯人工製品',
                'best_for': '科學分析，需要不確定性'
            },
            'Random Forest': {
                'visual': '塊狀結構',
                'smoothness': '局部平滑',
                'extremes': '良好保留',
                'artifacts': '矩形塊狀模式',
                'best_for': '複雜模式，特徵工程'
            }
        }
        
        print(f"{'方法':<15} {'視覺特徵':<12} {'平滑度':<8} {'極端值':<8} {'人工製品':<12} {'適用場景':<15}")
        print(f"{'-'*80}")
        
        for method, analysis in method_analysis.items():
            if any(method in title for title, _ in methods_data):
                print(f"{method:<15} {analysis['visual']:<12} {analysis['smoothness']:<8} "
                      f"{analysis['extremes']:<8} {analysis['artifacts']:<12} {analysis['best_for']:<15}")
        
        # ─── 物理合理性評估 ───
        print(f"\n🌪️ 物理合理性評估 (颱風降雨):")
        
        print(f"1. Nearest Neighbor:")
        print(f"   ❌ 不符合颱風降雨的連續性特徵")
        print(f"   ❌ 忽略了測站間的漸變過程")
        
        print(f"\n2. IDW:")
        print(f"   ⚠️ 部分合理，但牛眼效應不自然")
        print(f"   ⚠️ 距離衰減過於簡化")
        
        print(f"\n3. Ordinary Kriging:")
        print(f"   ✅ 最符合物理連續性")
        print(f"   ✅ 考慮空間相關性結構")
        print(f"   ✅ 提供不確定性評估")
        
        print(f"\n4. Random Forest:")
        print(f"   ⚠️ 可以學習複雜模式，但可能過擬合")
        print(f"   ⚠️ 塊狀結構不太符合氣象原理")
        print(f"   ✅ 能保留極端值，有利於預警")
        
        # ─── 計算效率比較 ───
        print(f"\n⏱️ 計算效率比較:")
        
        efficiency_data = []
        if 'kriging_results' in globals():
            efficiency_data.append(('Kriging', '較慢', '需要 variogram 擬合'))
        if 'rf_results' in globals():
            efficiency_data.append(('Random Forest', f"{rf_results['prediction_time']:.1f}s", '訓練慢，預測快'))
        if 'baseline_results' in globals():
            efficiency_data.append(('Nearest Neighbor', f"{baseline_results['nn_time']:.1f}s", '非常快'))
            efficiency_data.append(('IDW', f"{baseline_results['idw_time']:.1f}s', '中等速度'))
        
        print(f"{'方法':<15} {'預測時間':<10} {'特徵':<20}")
        print(f"{'-'*50}")
        for method, time_cost, feature in efficiency_data:
            print(f"{method:<15} {time_cost:<10} {feature:<20}")
        
        # ─── 指揮官決策建議 ───
        print(f"\n⚡ 指揮官決策建議:")
        print(f"🎯 緊急疏散決策:")
        print(f"   首選: Kriging (平滑 + 不確定性)")
        print(f"   備選: Random Forest (保留極端值)")
        
        print(f"\n🎯 快速評估決策:")
        print(f"   首選: Nearest Neighbor (速度最快)")
        print(f"   備選: IDW (平衡速度與品質)")
        
        print(f"\n🎯 科學分析決策:")
        print(f"   首選: Kriging (統計基礎最強)")
        print(f"   備選: Random Forest (可解釋性)")
        
        print(f"\n💾 四種方法比較圖已儲存為 {output_filename}")
        print(f"🚀 準備進行 Kriging vs RF 直接比較")
        
else:
    print("❌ 網格資訊不存在，請先執行前面的 Cell")

In [ ]:
# YOUR CODE HERE:
# 1. Create fig, axes = plt.subplots(2, 2, figsize=(18, 14))
# 2. Plot all four methods with imshow
# 3. Use extent=[x_min, x_max, y_min, y_max], origin='lower'
# 4. Overlay station scatter points
# 5. Add colorbars, titles, save figure

# vmax = max(z) * 1.1
# methods = [
#     ('Nearest Neighbor\n(Voronoi / "Patchwork")', z_nn),
#     ('IDW\n(Bullseye Effect)', z_idw),
#     ('Ordinary Kriging\n(Smooth + Sigma Map)', z_kriging),
#     ('Random Forest\n(ML "Block" Artifacts)', z_rf),
# ]

# for ax, (title, data) in zip(axes.flatten(), methods):
#     im = ax.imshow(data, extent=[x_min, x_max, y_min, y_max],
#                    origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
#     ax.scatter(x, y, c='black', s=8, zorder=5)
#     ax.set_title(title, fontsize=12, fontweight='bold')
#     plt.colorbar(im, ax=ax, shrink=0.7, label='mm/hr')

# plt.suptitle('Typhoon Fung-wong — Four Interpolation Methods Compared', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.savefig('interpolation_shootout.png', dpi=150, bbox_inches='tight')
# plt.show()

# Kriging vs Random Forest - 直接比較
# 三面板比較：Kriging | RF | Difference Map

if 'kriging_results' in globals() and 'rf_results' in globals():
    print("⚖️ Kriging vs Random Forest - 直接比較")
    print("🗺️ 差異地圖揭示兩種方法意見分歧的區域")
    
    z_kriging = kriging_results['z_kriging']
    z_rf = rf_results['z_rf']
    
    # 計算差異地圖
    diff = z_kriging - z_rf
    
    print(f"📊 差異分析:")
    print(f"  差異範圍: {diff.min():.2f} - {diff.max():.2f} mm/hr")
    print(f"  平均差異: {diff.mean():.2f} mm/hr")
    print(f"  差異標準差: {diff.std():.2f} mm/hr")
    
    # 計算統計指標
    mae = np.mean(np.abs(diff))  # Mean Absolute Error
    rmse = np.sqrt(np.mean(diff**2))  # Root Mean Square Error
    
    print(f"  MAE (平均絕對誤差): {mae:.2f} mm/hr")
    print(f"  RMSE (均方根誤差): {rmse:.2f} mm/hr")
    
    # 相關性分析
    correlation = np.corrcoef(z_kriging.flatten(), z_rf.flatten())[0, 1]
    print(f"  相關係數: {correlation:.3f}")
    
    # 建立 3 面板比較圖
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle('Kriging vs Random Forest - Head-to-Head Comparison', 
                 fontsize=16, fontweight='bold')
    
    extent = [grid_info['x_min']/1000, grid_info['x_max']/1000, 
              grid_info['y_min']/1000, grid_info['y_max']/1000]
    
    # 統一色彩範圍
    vmax = max(z_kriging.max(), z_rf.max()) * 1.1
    
    # ─── 面板 1: Kriging ───
    im1 = axes[0].imshow(z_kriging, extent=extent, origin='lower', 
                        cmap='YlOrRd', vmin=0, vmax=vmax)
    axes[0].scatter(x/1000, y/1000, c='black', s=10, zorder=5, alpha=0.7)
    axes[0].set_title('Ordinary Kriging\n(Statistical + Uncertainty)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Eastings (km)')
    axes[0].set_ylabel('Northings (km)')
    axes[0].grid(True, alpha=0.3)
    cbar1 = plt.colorbar(im1, ax=axes[0], shrink=0.8)
    cbar1.set_label('mm/hr')
    
    # 添加統計資訊
    stats1 = f'Max: {z_kriging.max():.1f}\nMean: {z_kriging.mean():.1f}'
    axes[0].text(0.02, 0.98, stats1, transform=axes[0].transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=9)
    
    # ─── 面板 2: Random Forest ───
    im2 = axes[1].imshow(z_rf, extent=extent, origin='lower', 
                        cmap='YlOrRd', vmin=0, vmax=vmax)
    axes[1].scatter(x/1000, y/1000, c='black', s=10, zorder=5, alpha=0.7)
    axes[1].set_title('Random Forest\n(Machine Learning)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Eastings (km)')
    axes[1].grid(True, alpha=0.3)
    cbar2 = plt.colorbar(im2, ax=axes[1], shrink=0.8)
    cbar2.set_label('mm/hr')
    
    # 添加統計資訊
    stats2 = f'Max: {z_rf.max():.1f}\nMean: {z_rf.mean():.1f}'
    axes[1].text(0.02, 0.98, stats2, transform=axes[1].transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=9)
    
    # ─── 面板 3: Difference Map ───
    # 使用 RdBu_r (紅-藍反轉) 色彩圖，正值=Kriging更高，負值=RF更高
    vmax_diff = max(abs(diff.min()), abs(diff.max()))
    im3 = axes[2].imshow(diff, extent=extent, origin='lower', 
                        cmap='RdBu_r', vmin=-vmax_diff, vmax=vmax_diff)
    axes[2].scatter(x/1000, y/1000, c='black', s=10, zorder=5, alpha=0.7)
    axes[2].set_title('Difference Map\n(Kriging - Random Forest)', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Eastings (km)')
    axes[2].grid(True, alpha=0.3)
    cbar3 = plt.colorbar(im3, ax=axes[2], shrink=0.8)
    cbar3.set_label('mm/hr')
    
    # 添加差異統計
    stats3 = f'Max: {diff.max():.1f}\nMin: {diff.min():.1f}\nMAE: {mae:.1f}'
    axes[2].text(0.02, 0.98, stats3, transform=axes[2].transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=9)
    
    plt.tight_layout()
    
    # 儲存圖片
    output_filename = 'kriging_vs_rf.png'
    plt.savefig(output_filename, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"✅ 圖片已儲存: {output_filename}")
    
    plt.show()
    
    # ─── 差異區域分析 ───
    print(f"\n🔍 差異區域分析:")
    
    # 定義顯著差異閾值
    significant_threshold = mae  # 使用 MAE 作為閾值
    
    high_kriging = diff > significant_threshold  # Kriging 顯著更高
    high_rf = diff < -significant_threshold  # RF 顯著更高
    
    print(f"顯著差異閾值: ±{significant_threshold:.1f} mm/hr")
    print(f"Kriging 更高區域: {np.sum(high_kriging):,} 點 ({np.sum(high_kriging)/diff.size*100:.1f}%)")
    print(f"Random Forest 更高區域: {np.sum(high_rf):,} 點 ({np.sum(high_rf)/diff.size*100:.1f}%)")
    print(f"一致區域: {np.sum(np.abs(diff) <= significant_threshold):,} 點 ({np.sum(np.abs(diff) <= significant_threshold)/diff.size*100:.1f}%)")
    
    # 分析高差異區域的特徵
    if np.any(high_kriging):
        kriging_dominant_rain = z_kriging[high_kriging]
        rf_dominant_rain = z_rf[high_kriging]
        
        print(f"\n📈 Kriging 主導區域特徵:")
        print(f"  平均降雨 (Kriging): {kriging_dominant_rain.mean():.1f} mm/hr")
        print(f"  平均降雨 (RF): {rf_dominant_rain.mean():.1f} mm/hr")
        print(f"  平均差異: {kriging_dominant_rain.mean() - rf_dominant_rain.mean():.1f} mm/hr")
    
    if np.any(high_rf):
        rf_dominant_rain = z_rf[high_rf]
        kriging_dominant_rain = z_kriging[high_rf]
        
        print(f"\n📊 Random Forest 主導區域特徵:")
        print(f"  平均降雨 (RF): {rf_dominant_rain.mean():.1f} mm/hr")
        print(f"  平均降雨 (Kriging): {kriging_dominant_rain.mean():.1f} mm/hr")
        print(f"  平均差異: {rf_dominant_rain.mean() - kriging_dominant_rain.mean():.1f} mm/hr")
    
    # ─── 決策含義分析 ───
    print(f"\n⚡ 決策含義分析:")
    
    print(f"🎯 高降雨 + 高差異區域:")
    high_rain_threshold = np.percentile(z, 75)  # 75th percentile
    high_rain_mask = (z_kriging > high_rain_threshold) | (z_rf > high_rain_threshold)
    high_disagreement = high_rain_mask & (np.abs(diff) > significant_threshold)
    
    if np.any(high_disagreement):
        disagreement_count = np.sum(high_disagreement)
        print(f"  高降雨差異點: {disagreement_count:,} 點")
        print(f"  佔高降雨區域比例: {disagreement_count/np.sum(high_rain_mask)*100:.1f}%")
        print(f"  ⚠️ 這些區域需要特別注意，方法選擇影響決策")
        
        # 分析這些區域的平均值
        kriging_in_disagreement = z_kriging[high_disagreement]
        rf_in_disagreement = z_rf[high_disagreement]
        
        print(f"  Kriging 平均: {kriging_in_disagreement.mean():.1f} mm/hr")
        print(f"  RF 平均: {rf_in_disagreement.mean():.1f} mm/hr")
        print(f"  最大差異: {np.abs(diff[high_disagreement]).max():.1f} mm/hr")
    else:
        print(f"  ✅ 高降雨區域兩種方法結果一致")
    
    # ─── 方法選擇建議 ───
    print(f"\n🎯 方法選擇建議:")
    
    if correlation > 0.8:
        print(f"📊 高相關性 ({correlation:.3f}): 兩種方法總體一致")
        print(f"   💡 可任選一種，Kriging 提供不確定性，RF 保留極端值")
    elif correlation > 0.6:
        print(f"📊 中等相關性 ({correlation:.3f}): 兩種方法有明顯差異")
        print(f"   💡 建議使用 Kriging 作為主要方法，RF 作為輔助驗證")
    else:
        print(f"📊 低相關性 ({correlation:.3f}): 兩種方法結果差異很大")
        print(f"   💡 需要謹慎選擇，考慮具體應用場景")
    
    if mae < 5:
        print(f"🎯 低平均誤差 ({mae:.1f} mm/hr): 兩種方法結果接近")
        print(f"   💡 選擇取決於是否需要不確定性評估")
    elif mae < 10:
        print(f"🎯 中等誤差 ({mae:.1f} mm/hr): 兩種方法有明顯差異")
        print(f"   💡 建議在差異較大區域部署額外監測")
    else:
        print(f"🎯 高誤差 ({mae:.1f} mm/hr): 兩種方法結果差異很大")
        print(f"   💡 需要外部驗證，可能需要結合其他資料")
    
    # 儲存比較結果
    global comparison_results
    comparison_results = {
        'diff': diff,
        'mae': mae,
        'rmse': rmse,
        'correlation': correlation,
        'significant_threshold': significant_threshold,
        'high_kriging_count': np.sum(high_kriging) if np.any(high_kriging) else 0,
        'high_rf_count': np.sum(high_rf) if np.any(high_rf) else 0
    }
    
    print(f"\n💾 比較結果已儲存")
    print(f"🚀 準備進行 Lab 1 反思")
    
else:
    print("❌ 缺少 Kriging 或 RF 結果，請先執行相關 Cell")

In [ ]:
# YOUR CODE HERE:
# 1. Create fig, axes = plt.subplots(1, 3, figsize=(22, 7))
# 2. Left: Kriging (YlOrRd)
# 3. Middle: Random Forest (YlOrRd)
# 4. Right: Difference (Kriging - RF) using RdBu_r colormap
# 5. Save as 'kriging_vs_rf.png'

# diff = z_kriging - z_rf
# ...

# Lab 1 反思

# 請回答以下問題：

"""
1. 哪種方法產生最物理合理的降雨模式？為什麼？

答案：Ordinary Kriging。原因：
- 基於空間統計理論，考慮空間相關性
- 提供不確定性評估（Sigma Map）
- 平滑自然，符合氣象物理原理
- 相比之下：NN 不連續、IDW 有牛眼效應、RF 有塊狀人工製品

2. Kriging 和 RF 在哪些區域差異最大？這告訴你什麼？

答案：在山區和測站稀疏區域差異最大。
這告訴我們：
- 這些區域預測不確定性高
- 需要額外監測或驗證
- 方法選擇影響決策結果
- 應該部署臨時監測設備

3. 觀察到哪些人工製品（artifacts）？

答案：
- Nearest Neighbor：Voronoi 補丁圖案，邊界不連續
- IDW：牛眼效應，測站周圍圓形模式
- Random Forest：塊狀結構，矩形人工模式
- Kriging：相對最少，但可能過度平滑

4. 如果你是指揮官，你會信任哪種方法進行疏散決策？為什麼？

答案：優先信任 Kriging，但參考 Random Forest。
原因：
- Kriging 提供不確定性資訊，決策更謹慎
- 統計基礎穩健，結果可解釋
- RF 保留極端值，有利於最壞情況評估
- 雙重驗證策略：Kriging 主要 + RF 輔助
"""

**Your Lab 1 reflection here:**

1. Most realistic: ...
2. Disagreement areas: ...
3. Artifacts: ...
4. Commander's choice: ...

# Lab 2: 信賴度與不確定性診斷
# The Sigma Map — Kriging 的獨特武器

if 'kriging_results' in globals():
    print("🎯 Sigma Map - Kriging 的獨特武器")
    print("🛡️ 這是讓 Kriging 特別的地方：沒有其他方法原生提供信心地圖")
    
    z_kriging = kriging_results['z_kriging']
    ss_kriging = kriging_results['ss_kriging']
    
    print(f"📊 Sigma Map 統計:")
    print(f"  方差範圍: {ss_kriging.min():.4f} - {ss_kriging.max():.4f}")
    print(f"  平均方差: {ss_kriging.mean():.4f}")
    print(f"  方差標準差: {ss_kriging.std():.4f}")
    
    # 建立雙面板圖：降雨估計 + 方差地圖
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('The Honest Report Card: Estimate + Confidence', 
                 fontsize=16, fontweight='bold')
    
    extent = [grid_info['x_min']/1000, grid_info['x_max']/1000, 
              grid_info['y_min']/1000, grid_info['y_max']/1000]
    
    # ─── 左面板：降雨估計 ───
    im1 = axes[0].imshow(z_kriging, extent=extent, origin='lower', 
                        cmap='YlOrRd', vmin=0, vmax=max(z)*1.1)
    axes[0].scatter(x/1000, y/1000, c='black', s=10, zorder=5, alpha=0.7)
    axes[0].set_title('Estimated Rainfall (mm/hr)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Eastings (km)')
    axes[0].set_ylabel('Northings (km)')
    axes[0].grid(True, alpha=0.3)
    cbar1 = plt.colorbar(im1, ax=axes[0], shrink=0.8)
    cbar1.set_label('mm/hr')
    
    # 添加降雨統計
    rain_stats = f'Max: {z_kriging.max():.1f}\nMean: {z_kriging.mean():.1f}'
    axes[0].text(0.02, 0.98, rain_stats, transform=axes[0].transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=9)
    
    # ─── 右面板：Kriging 方差（Sigma Map）──
    im2 = axes[1].imshow(ss_kriging, extent=extent, origin='lower', 
                        cmap='Blues', vmin=0)
    axes[1].scatter(x/1000, y/1000, c='red', s=10, zorder=5, alpha=0.7, label='Stations')
    axes[1].set_title('Kriging Sigma Map (Uncertainty)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Eastings (km)')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='upper right')
    cbar2 = plt.colorbar(im2, ax=axes[1], shrink=0.8)
    cbar2.set_label('Variance')
    
    # 添加方差統計
    var_stats = f'Max: {ss_kriging.max():.4f}\nMean: {ss_kriging.mean():.4f}'
    axes[1].text(0.02, 0.98, var_stats, transform=axes[1].transAxes,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top', fontsize=9)
    
    plt.tight_layout()
    
    # 儲存圖片
    output_filename = 'sigma_map.png'
    plt.savefig(output_filename, dpi=150, bbox_inches='tight', facecolor='white')
    print(f"✅ 圖片已儲存: {output_filename}")
    
    plt.show()
    
    # ─── 信賴度分級分析 ───
    print(f"\n🎯 信賴度分級分析:")
    
    # 計算信賴度分級閾值
    var_33 = np.percentile(ss_kriging, 33)
    var_66 = np.percentile(ss_kriging, 66)
    
    print(f"信賴度閾值:")
    print(f"  HIGH 信賴度: variance < {var_33:.4f} (低方差)")
    print(f"  MEDIUM 信賴度: {var_33:.4f} ≤ variance < {var_66:.4f}")
    print(f"  LOW 信賴度: variance ≥ {var_66:.4f} (高方差)")
    
    # 分類信賴度
    high_confidence = ss_kriging < var_33
    medium_confidence = (ss_kriging >= var_33) & (ss_kriging < var_66)
    low_confidence = ss_kriging >= var_66
    
    print(f"\n📊 信賴度分佈:")
    print(f"  HIGH 信賴度: {np.sum(high_confidence):,} 點 ({np.sum(high_confidence)/ss_kriging.size*100:.1f}%)")
    print(f"  MEDIUM 信賴度: {np.sum(medium_confidence):,} 點 ({np.sum(medium_confidence)/ss_kriging.size*100:.1f}%)")
    print(f"  LOW 信賴度: {np.sum(low_confidence):,} 點 ({np.sum(low_confidence)/ss_kriging.size*100:.1f}%)")
    
    # ─── 決策支援分析 ───
    print(f"\n⚡ 決策支援分析:")
    
    # 高降雨 + 低信賴度區域（最危險）
    high_rain_threshold = np.percentile(z, 75)
    high_rain_mask = z_kriging > high_rain_threshold
    
    dangerous_combinations = {
        'confirmed_high_risk': high_rain_mask & high_confidence,
        'uncertain_high_risk': high_rain_mask & low_confidence,
        'medium_risk': high_rain_mask & medium_confidence
    }
    
    print(f"🚨 高降雨區域決策分析 (閾值: >{high_rain_threshold:.1f} mm/hr):")
    
    for risk_type, mask in dangerous_combinations.items():
        count = np.sum(mask)
        if count > 0:
            avg_rain = z_kriging[mask].mean()
            avg_var = ss_kriging[mask].mean()
            
            if risk_type == 'confirmed_high_risk':
                print(f"  ✅ CONFIRMED: {count:,} 點")
                print(f"     平均降雨: {avg_rain:.1f} mm/hr")
                print(f"     平均方差: {avg_var:.4f} (低)")
                print(f"     🎯 決策: 立即疏散，高信心預警")
                
            elif risk_type == 'uncertain_high_risk':
                print(f"  ⚠️ UNCERTAIN: {count:,} 點")
                print(f"     平均降雨: {avg_rain:.1f} mm/hr")
                print(f"     平均方差: {avg_var:.4f} (高)")
                print(f"     🎯 決策: 部署臨時監測，準備疏散")
                
            else:  # medium_risk
                print(f"  🔍 MEDIUM: {count:,} 點")
                print(f"     平均降雨: {avg_rain:.1f} mm/hr")
                print(f"     平均方差: {avg_var:.4f} (中)")
                print(f"     🎯 決策: 持續監測，準備預警")
    
    # ─── 測站密度與信賴度關係 ───
    print(f"\n📍 測站密度與信賴度關係:")
    
    # 計算每個網格點到最近測站的距離
    from scipy.spatial.distance import cdist
    grid_points = np.column_stack([grid_info['grid_xx'].ravel(), grid_info['grid_yy'].ravel()])
    station_points = np.column_stack([x, y])
    
    distances = cdist(grid_points, station_points).min(axis=1)
    distances = distances.reshape(grid_info['shape'])
    
    # 分析距離與方差的關係
    distance_bins = [0, 2000, 5000, 10000, 20000, 50000]  # 公尺
    distance_labels = ['<2km', '2-5km', '5-10km', '10-20km', '20-50km', '>50km']
    
    print(f"距離範圍\t平均方差\t平均降雨\t網格點數")
    print(f"-" * 50)
    
    for i in range(len(distance_bins)-1):
        mask = (distances >= distance_bins[i]) & (distances < distance_bins[i+1])
        if np.any(mask):
            avg_var = ss_kriging[mask].mean()
            avg_rain = z_kriging[mask].mean()
            count = np.sum(mask)
            print(f"{distance_labels[i]:<8}\t{avg_var:.4f}\t\t{avg_rain:.2f}\t\t{count:,}")
    
    # 最後一個區間
    mask = distances >= distance_bins[-1]
    if np.any(mask):
        avg_var = ss_kriging[mask].mean()
        avg_rain = z_kriging[mask].mean()
        count = np.sum(mask)
        print(f"{distance_labels[-1]:<8}\t{avg_var:.4f}\t\t{avg_rain:.2f}\t\t{count:,}")
    
    # ─── 指揮官決策矩陣 ───
    print(f"\n🎯 指揮官決策矩陣:")
    print(f"┌─────────────────┬─────────────────┬─────────────────┐")
    print(f"│ 降雨量 \ 信賴度  │   HIGH (低方差)  │  LOW (高方差)   │")
    print(f"├─────────────────┼─────────────────┼─────────────────┤")
    print(f"│ HIGH (>75th%)   │ 立即疏散        │ 部署監測      │")
    print(f"│                 │ 確認高風險      │ 準備疏散        │")
    print(f"├─────────────────┼─────────────────┼─────────────────┤")
    print(f"│ MEDIUM (50-75%) │ 準備預警        │ 持續監測      │")
    print(f"│                 │ 通知相關單位    │ 收集更多資料    │")
    print(f"├─────────────────┼─────────────────┼─────────────────┤")
    print(f"│ LOW (<50th%)    │ 持續監測        │ 正常監測      │")
    print(f"│                 │ 定期更新        │ 無需特別行動  │")
    print(f"└─────────────────┴─────────────────┴─────────────────┘")
    
    # 儲存 Sigma Map 分析結果
    global sigma_analysis
    sigma_analysis = {
        'var_33': var_33,
        'var_66': var_66,
        'high_confidence_count': np.sum(high_confidence),
        'low_confidence_count': np.sum(low_confidence),
        'confirmed_high_risk': np.sum(dangerous_combinations['confirmed_high_risk']),
        'uncertain_high_risk': np.sum(dangerous_combinations['uncertain_high_risk'])
    }
    
    print(f"\n💾 Sigma Map 分析結果已儲存")
    print(f"🚀 準備進行 Nugget Effect 實驗")
    
else:
    print("❌ Kriging 結果不存在，請先執行 Kriging Cell")

In [ ]:
# YOUR CODE HERE:
# Create a 2-panel figure:
# Left: z_kriging with YlOrRd colormap (rainfall estimate)
# Right: ss_kriging with Blues colormap (variance/uncertainty)
# Add station locations on both panels (red dots on variance map)
# Save as 'sigma_map.png'

# fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# # Rainfall estimate
# im1 = axes[0].imshow(z_kriging, extent=[x_min, x_max, y_min, y_max],
#                       origin='lower', cmap='YlOrRd', vmin=0)
# axes[0].scatter(x, y, c='black', s=10, zorder=5)
# axes[0].set_title('Estimated Rainfall (mm/hr)')
# plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

# # Kriging Variance (Sigma Map)
# im2 = axes[1].imshow(ss_kriging, extent=[x_min, x_max, y_min, y_max],
#                       origin='lower', cmap='Blues', vmin=0)
# axes[1].scatter(x, y, c='red', s=10, zorder=5, label='Stations')
# axes[1].set_title('Kriging Sigma Map (Uncertainty)')
# axes[1].legend(loc='upper right')
# plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Variance')

# plt.suptitle('The Honest Report Card: Estimate + Confidence', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.savefig('sigma_map.png', dpi=150, bbox_inches='tight')
# plt.show()

# print(f"Variance range: {np.nanmin(ss_kriging):.1f} - {np.nanmax(ss_kriging):.1f}")

# Nugget Effect — 為什麼極端降雨被稀釋
# 比較 Nugget=10% vs Nugget=1% 在最大降雨站周圍的效果

if x is not None and y is not None and z is not None:
    print("🔬 Nugget Effect 實驗")
    print("🎯 為什麼蘇澳站 130.5 mm/hr 在 500m 外預測只有 ~71 mm？")
    
    # 找到最大降雨站
    suao_idx = np.argmax(z)
    suao_x, suao_y, suao_z = x[suao_idx], y[suao_idx], z[suao_idx]
    
    print(f"🌪️ 最大降雨站分析:")
    print(f"  測站索引: {suao_idx}")
    print(f"  座標: ({suao_x/1000:.1f}, {suao_y/1000:.1f}) km")
    print(f"  實際降雨: {suao_z:.1f} mm/hr")
    
    # 找到對應的測站名稱
    station_name = study_rain_3826.iloc[suao_idx]['station_name']
    print(f"  測站名稱: {station_name}")
    
    # 建立局部網格（20km × 20km）
    local_range = 20000  # 20km
    local_buffer = 2000  # 2km
    
    local_x_min = suao_x - local_range/2
    local_x_max = suao_x + local_range/2
    local_y_min = suao_y - local_range/2
    local_y_max = suao_y + local_range/2
    
    # 建立高解析度局部網格
    local_resolution = 200  # 200m 解析度
    local_grid_x = np.arange(local_x_min - local_buffer, local_x_max + local_buffer, local_resolution)
    local_grid_y = np.arange(local_y_min - local_buffer, local_y_max + local_buffer, local_resolution)
    
    print(f"\n🗺️ 局部網格設定:")
    print(f"  範圍: {local_range/1000:.0f}km × {local_range/1000:.0f}km")
    print(f  解析度: {local_resolution}m")
    print(f"  網格大小: {len(local_grid_x)}×{len(local_grid_y)} = {len(local_grid_x)*len(local_grid_y):,} 點")
    
    # 建立兩個不同 Nugget 設定的模型
    sill_val = float(z_log_global.var())
    
    print(f"\n🔧 模型參數:")
    print(f"  Sill: {sill_val:.3f}")
    print(f"  Nugget (高): {sill_val * 0.10:.3f} (10%)")
    print(f"  Nugget (低): {sill_val * 0.01:.3f} (1%)")
    
    # ─── 高 Nugget 模型 (10%) ───
    print(f"\n🔴 高 Nugget 模型 (10%):")
    OK_high_nugget = OrdinaryKriging(x, y, z_log_global, variogram_model='spherical',
                                    verbose=False, enable_plotting=False, nlags=15,
                                    variogram_parameters={
                                        'sill': sill_val,
                                        'range': 50000.0,
                                        'nugget': sill_val * 0.10
                                    })
    
    # ─── 低 Nugget 模型 (1%) ───
    print(f"🟢 低 Nugget 模型 (1%):")
    OK_low_nugget = OrdinaryKriging(x, y, z_log_global, variogram_model='spherical',
                                   verbose=False, enable_plotting=False, nlags=15,
                                   variogram_parameters={
                                       'sill': sill_val,
                                       'range': 50000.0,
                                       'nugget': sill_val * 0.01
                                   })
    
    # 在局部網格上預測
    print(f"\n⏱️ 執行局部預測...")
    
    # 高 Nugget 預測
    z_high_log, ss_high_log = OK_high_nugget.execute('grid', local_grid_x, local_grid_y)
    z_high = np.expm1(z_high_log)
    z_high[z_high < 0] = 0
    
    # 低 Nugget 預測
    z_low_log, ss_low_log = OK_low_nugget.execute('grid', local_grid_x, local_grid_y)
    z_low = np.expm1(z_low_log)
    z_low[z_low < 0] = 0
    
    print(f"✅ 預測完成")
    
    # ─── 視覺化比較 ───
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.suptitle(f'Nugget Effect Comparison - {station_name} ({suao_z:.1f} mm/hr)', 
                 fontsize=14, fontweight='bold')
    
    extent = [local_x_min/1000, local_x_max/1000, local_y_min/1000, local_y_max/1000]
    vmax = suao_z * 1.2
    
    # 高 Nugget 圖
    im1 = axes[0].imshow(z_high, extent=extent, origin='lower', 
                        cmap='YlOrRd', vmin=0, vmax=vmax)
    axes[0].scatter(x/1000, y/1000, c='black', s=15, zorder=5, alpha=0.7)
    axes[0].scatter(suao_x/1000, suao_y/1000, c='red', s=100, marker='*', zorder=6, 
                   label=f'{station_name} ({suao_z:.1f} mm/hr)')
    axes[0].set_title('High Nugget (10%)\n"Measurements are noisy"', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Eastings (km)')
    axes[0].set_ylabel('Northings (km)')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')
    
    # 低 Nugget 圖
    im2 = axes[1].imshow(z_low, extent=extent, origin='lower', 
                        cmap='YlOrRd', vmin=0, vmax=vmax)
    axes[1].scatter(x/1000, y/1000, c='black', s=15, zorder=5, alpha=0.7)
    axes[1].scatter(suao_x/1000, suao_y/1000, c='red', s=100, marker='*', zorder=6,
                   label=f'{station_name} ({suao_z:.1f} mm/hr)')
    axes[1].set_title('Low Nugget (1%)\n"Measurements are accurate"', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Eastings (km)')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.colorbar(im2, ax=axes[1], shrink=0.8, label='mm/hr')
    
    plt.tight_layout()
    plt.show()
    
    # ─── 距離衰減分析 ───
    print(f"\n📏 距離衰減分析:")
    
    # 計算不同距離的預測值
    test_distances = [0, 500, 1000, 2000, 5000]  # 公尺
    
    print(f"{'距離':<8} {'高Nugget':<10} {'低Nugget':<10} {'差異':<8} {'保留率(高)':<10} {'保留率(低)':<10}")
    print(f"{'-' * 60}")
    
    for dist in test_distances:
        # 計算該距離處的預測值
        if dist == 0:
            pred_high = suao_z
            pred_low = suao_z
        else:
            # 在四個方向取平均
            angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
            preds_high = []
            preds_low = []
            
            for angle in angles:
                test_x = suao_x + dist * np.cos(angle)
                test_y = suao_y + dist * np.sin(angle)
                
                # 高 Nugget 預測
                pred_high_log, _ = OK_high_nugget.execute('points', [test_x], [test_y])
                pred_high_val = np.expm1(pred_high_log[0])
                preds_high.append(pred_high_val)
                
                # 低 Nugget 預測
                pred_low_log, _ = OK_low_nugget.execute('points', [test_x], [test_y])
                pred_low_val = np.expm1(pred_low_log[0])
                preds_low.append(pred_low_val)
            
            pred_high = np.mean(preds_high)
            pred_low = np.mean(preds_low)
        
        diff = pred_high - pred_low
        retention_high = pred_high / suao_z * 100
        retention_low = pred_low / suao_z * 100
        
        print(f"{dist:<8}m {pred_high:<10.1f} {pred_low:<10.1f} {diff:<8.1f} {retention_high:<10.1f}% {retention_low:<10.1f}%")
    
    # ─── 解讀與建議 ───
    print(f"\n🔍 解讀與建議:")
    
    print(f"📊 觀察結果:")
    print(f"  • 高 Nugget (10%): 測量值被視為「噪音」，預測更平滑")
    print(f"  • 低 Nugget (1%): 測量值被視為「準確」，預測更貼近測站值")
    
    print(f"\n🎯 對 CWA 校準測站的建議:")
    print(f"  ✅ 推薦使用低 Nugget (1-5%)")
    print(f"  💡 理由: CWA 測站經過校準，測量品質高")
    print(f"  💡 好處: 更好地保留極端降雨值，有利於預警")
    
    print(f"\n⚠️ 何時使用高 Nugget:")
    print(f"  🌡️ 當測站品質不確定時")
    print(f"  📡 當儀器精度較低時")
    print(f"  🌊 當測站環境干擾嚴重時")
    print(f"  🔬 當需要更保守的預測時")
    
    print(f"\n🚨 實務影響:")
    print(f"  高 Nugget 可能低估極端事件 → 預警延遲")
    print(f"  低 Nugget 可能高估雜訊 → 錯誤警報")
    print(f"  💡 需要在靈敏性和特異性之間平衡")
    
    # 儲存實驗結果
    global nugget_experiment
    nugget_experiment = {
        'station_name': station_name,
        'suao_z': suao_z,
        'suao_x': suao_x,
        'suao_y': suao_y,
        'high_nugget_predictions': z_high,
        'low_nugget_predictions': z_low,
        'high_nugget_value': sill_val * 0.10,
        'low_nugget_value': sill_val * 0.01
    }
    
    print(f"\n💾 Nugget Effect 實驗結果已儲存")
    print(f"🚀 準備進行 GeoTIFF 輸出")
    
else:
    print("❌ 無效資料，請先執行前面的 Cell")

In [ ]:
# YOUR CODE HERE:
# 1. Find the station with maximum rainfall (suao_idx = np.argmax(z))
# 2. Create two OrdinaryKriging models: nugget = sill*0.10 and sill*0.01
# 3. Predict on a local grid (20km box) around that station
# 4. Plot side-by-side comparison maps
# 5. Predict at specific offsets: 0m, 500m, 1000m, 2000m from the station
# 6. Print comparison table

# suao_idx = np.argmax(z)
# suao_x, suao_y, suao_z = x[suao_idx], y[suao_idx], z[suao_idx]
# print(f"Station with max rainfall: {suao_z:.1f} mm/hr")

# sill_val = float(z_log.var())
# ... create OK with nugget = sill_val * 0.10
# ... create OK with nugget = sill_val * 0.01
# ... compare predictions

# 🔑 Which Nugget setting is better for CWA calibrated stations? Why?

import rasterio
from rasterio.transform import from_bounds

# 輸出到 GeoTIFF
# ⚠️ 翻轉警告：z_kriging row 0 = 南邊 (numpy 慣例)。GeoTIFF row 0 = 北邊。使用 np.flipud() 修正。

if 'grid_info' in globals():
    print("💾 輸出到 GeoTIFF")
    print("🗺️ 將 2D numpy 陣列儲存為地理資訊系統可讀的 GeoTIFF 格式")
    
    # 計算 rasterio transform
    transform = from_bounds(
        grid_info['x_min'], grid_info['y_min'], 
        grid_info['x_max'], grid_info['y_max'],
        width=grid_info['shape'][1], 
        height=grid_info['shape'][0]
    )
    
    print(f"📐 GeoTIFF 參數:")
    print(f"  座標參考系統: EPSG:3826")
    print(f"  資料型態: float32")
    print(f"  影像尺寸: {grid_info['shape'][0]} × {grid_info['shape'][1]}")
    print(f"  解析度: {grid_info['resolution']} m")
    print(f"  範圍: {grid_info['x_min']/1000:.1f} - {grid_info['x_max']/1000:.1f} km (E)")
    print(f"        {grid_info['y_min']/1000:.1f} - {grid_info['y_max']/1000:.1f} km (N)")
    
    def save_geotiff(data, filename, crs='EPSG:3826', nodata=-9999):
        """儲存 GeoTIFF 檔案"""
        try:
            # ⚠️ 重要：翻轉 Y 軸以符合 GeoTIFF 慣例
            data_flipped = np.flipud(data).astype(np.float32)
            
            # 建立輸出目錄
            import os
            os.makedirs('output', exist_ok=True)
            
            with rasterio.open(
                f'output/{filename}', 'w',
                driver='GTiff',
                height=data_flipped.shape[0],
                width=data_flipped.shape[1],
                count=1,
                dtype=data_flipped.dtype,
                crs=crs,
                transform=transform,
                nodata=nodata
            ) as dst:
                dst.write(data_flipped, 1)
            
            # 檢查檔案大小
            file_size = os.path.getsize(f'output/{filename}') / (1024*1024)  # MB
            print(f"✅ 已儲存: {filename} ({file_size:.1f} MB)")
            
            return True
            
        except Exception as e:
            print(f"❌ 儲存失敗 {filename}: {e}")
            return False
    
    # ─── 輸出 Kriging 降雨 ───
    print(f"\n🌧️ 輸出 Kriging 降雨 GeoTIFF...")
    if 'kriging_results' in globals():
        success1 = save_geotiff(kriging_results['z_kriging'], 'kriging_rainfall.tif')
        
        if success1:
            # 顯示統計資訊
            data = kriging_results['z_kriging']
            print(f"   資料範圍: {data.min():.2f} - {data.max():.2f} mm/hr")
            print(f"   平均值: {data.mean():.2f} mm/hr")
            print(f"   非零像素: {np.sum(data > 0):,} ({np.sum(data > 0)/data.size*100:.1f}%)")
    else:
        print(f"❌ Kriging 結果不存在")
        success1 = False
    
    # ─── 輸出 Kriging 方差 ───
    print(f"\n📊 輸出 Kriging 方差 GeoTIFF...")
    if 'kriging_results' in globals():
        success2 = save_geotiff(kriging_results['ss_kriging'], 'kriging_variance.tif')
        
        if success2:
            # 顯示統計資訊
            data = kriging_results['ss_kriging']
            print(f"   方差範圍: {data.min():.4f} - {data.max():.4f}")
            print(f"   平均方差: {data.mean():.4f}")
            print(f"   高方差像素: {np.sum(data > np.percentile(data, 66)):,}")
    else:
        print(f"❌ Kriging 方差不存在")
        success2 = False
    
    # ─── 輸出 Random Forest 降雨 ───
    print(f"\n🤖 輸出 Random Forest 降雨 GeoTIFF...")
    if 'rf_results' in globals():
        success3 = save_geotiff(rf_results['z_rf'], 'rf_rainfall.tif')
        
        if success3:
            # 顯示統計資訊
            data = rf_results['z_rf']
            print(f"   資料範圍: {data.min():.2f} - {data.max():.2f} mm/hr")
            print(f"   平均值: {data.mean():.2f} mm/hr")
            print(f"   非零像素: {np.sum(data > 0):,} ({np.sum(data > 0)/data.size*100:.1f}%)")
    else:
        print(f"❌ Random Forest 結果不存在")
        success3 = False
    
    # ─── 輸出摘要 ───
    print(f"\n📋 輸出摘要:")
    print(f"{'檔案名稱':<20} {'狀態':<8} {'大小(MB)':<10} {'描述'}")
    print(f"{'-' * 60}")
    
    files_info = [
        ('kriging_rainfall.tif', success1, 'Kriging 降雨估計'),
        ('kriging_variance.tif', success2, 'Kriging 不確定性'),
        ('rf_rainfall.tif', success3, 'Random Forest 降雨')
    ]
    
    total_size = 0
    successful_files = 0
    
    for filename, success, description in files_info:
        if success:
            try:
                file_path = f'output/{filename}'
                file_size = os.path.getsize(file_path) / (1024*1024)
                status = "✅ 成功"
                total_size += file_size
                successful_files += 1
            except:
                file_size = 0
                status = "❌ 失敗"
        else:
            file_size = 0
            status = "❌ 失敗"
        
        print(f"{filename:<20} {status:<8} {file_size:<10.1f} {description}")
    
    print(f"{'-' * 60})
    print(f"總計: {successful_files}/3 檔案成功, {total_size:.1f} MB")
    
    # ─── GeoTIFF 驗證 ───
    print(f"\n🔍 GeoTIFF 驗證:")
    
    if successful_files > 0:
        print(f"✅ 檢查輸出檔案...")
        
        try:
            # 驗證第一個成功檔案
            for filename, success, _ in files_info:
                if success:
                    file_path = f'output/{filename}'
                    with rasterio.open(file_path) as src:
                        print(f"📄 {filename}:")
                        print(f"   CRS: {src.crs}")
                        print(f"   Transform: {src.transform}")
                        print(f"   尺寸: {src.width} × {src.height}")
                        print(f"   資料型態: {src.dtypes[0]}")
                        print(f"   座標範圍: {src.bounds}")
                        break  # 只檢查第一個檔案
                        
        except Exception as e:
            print(f"❌ 驗證失敗: {e}")
    
    # ─── 使用說明 ───
    print(f"\n📖 GeoTIFF 使用說明:")
    print(f"🗺️ QGIS:")
    print(f"   1. 開啟 QGIS")
    print(f"   2. 點選「圖層」→「新增圖層」→「新增 raster 圖層」")
    print(f"   3. 選擇輸出的 .tif 檔案")
    print(f"   4. 設定正確的 CRS (EPSG:3826)")
    
    print(f"\n🐍 Python (rasterio):")
    print(f"   import rasterio")
    print(f"   with rasterio.open('output/kriging_rainfall.tif') as src:")
    print(f"       data = src.read(1)")
    print(f"       transform = src.transform")
    print(f"       crs = src.crs")
    
    print(f"\n🌐 ArcGIS:")
    print(f"   1. 開啟 ArcMap 或 ArcGIS Pro")
    print(f"   2. 新增資料 → 新增 raster 資料")
    print(f"   3. 選擇輸出的 .tif 檔案")
    print(f"   4. 設定投影坐標系統")
    
    # ─── 後續分析建議 ───
    print(f"\n💡 後續分析建議:")
    print(f"🔍 空間分析:")
    print(f"   • 使用 rasterio 進行像素統計")
    print(f"   • 與其他 GIS 圖層疊合分析")
    print(f"   • 計算不同降雨等級的面積")
    
    print(f"\n📊 區域統計:")
    print(f"   • 使用 rasterstats 進行 zonal statistics")
    print(f"   • 結合行政區界線進行統計分析")
    print(f"   • 生成決策支援表格")
    
    print(f"\n🌐 網路地圖:")
    print(f"   • 使用 folium 或 leaflet 發布互動地圖")
    print(f"   • 轉換為瓦片圖層")
    print(f"   • 整合到 Web GIS 平台")
    
    print(f"\n💾 檔案管理:")
    print(f"   • 所有檔案儲存在 output/ 目錄")
    print(f"   • 檔案格式：GeoTIFF (壓縮建議 LZW)")
    print(f"   • 備份：建議保留原始 numpy 檔案")
    
    print(f"\n✅ GeoTIFF 輸出完成")
    print(f"🚀 準備進行區域統計分析")
    
else:
    print("❌ 網格資訊不存在，請先執行前面的 Cell")

In [ ]:
import rasterio
from rasterio.transform import from_bounds

# YOUR CODE HERE:
# 1. Compute rasterio transform using from_bounds
# 2. Write a helper function save_geotiff(data, filename)
# 3. Save kriging_rainfall.tif, kriging_variance.tif, rf_rainfall.tif
# Remember: np.flipud() before writing!

# transform = from_bounds(x_min, y_min, x_max, y_max,
#                         width=z_kriging.shape[1], height=z_kriging.shape[0])

# def save_geotiff(data, filename, crs='EPSG:3826'):
#     data_flipped = np.flipud(data).astype(np.float32)
#     with rasterio.open(filename, 'w', driver='GTiff',
#         height=data_flipped.shape[0], width=data_flipped.shape[1],
#         count=1, dtype='float32', crs=crs, transform=transform, nodata=-9999
#     ) as dst:
#         dst.write(data_flipped, 1)
#     print(f"✓ Saved {filename}")

# save_geotiff(z_kriging, 'kriging_rainfall.tif')
# save_geotiff(ss_kriging, 'kriging_variance.tif')
# save_geotiff(z_rf, 'rf_rainfall.tif')

# 區域統計 - 鄉鎮決策表格
# 計算各鄉鎮的統計資料並比較 Kriging 和 RF 的結果

from rasterstats import zonal_stats

print("🏘️ 區域統計 - 鄉鎮決策表格")
print("📊 計算各鄉鎮的降雨統計並比較方法差異")

# ─── 嘗試載入鄉鎮界線 ───
print(f"\n🔍 嘗試載入鄉鎮界線資料...")

# 嘗試多種可能的檔案路徑
possible_paths = [
    'TOWN_MOI.shp',
    'data/TOWN_MOI.shp', 
    '../data/TOWN_MOI.shp',
    '../../week5/data/TOWN_MOI.shp'
]

towns = None
for path in possible_paths:
    try:
        towns = gpd.read_file(path)
        print(f"✅ 成功載入: {path}")
        break
    except:
        continue

if towns is None:
    print(f"❌ 鄉鎮界線檔案不存在")
    print(f"📝 預期輸出格式說明:")
    print(f"如果檔案存在，預期輸出如下表格:")
    
    # 建立預期輸出範例
    expected_output = """
┌─────────────────────┬──────────┬──────────┬──────────┬──────────┬─────────────┐
│ 鄉鎮              │ 縣市      │ Kriging平均 │ Kriging最大 │ RF平均    │ 平均variance │ 可信度    │
├─────────────────────┼──────────┼──────────┼──────────┼──────────┼─────────────┼───────────┤
│ 花蓮市            │ 花蓮縣    │ 15.2     │ 45.8      │ 16.1     │ 0.0234      │ HIGH     │
│ 吉安鄉            │ 花蓮縣    │ 12.8     │ 38.2      │ 13.5     │ 0.0456      │ MEDIUM   │
│ 壽豐鄉            │ 花蓮縣    │ 8.5      │ 25.6      │ 9.2      │ 0.0789      │ LOW      │
│ 宜蘭市            │ 宜蘭縣    │ 18.7     │ 52.3      │ 19.8     │ 0.0198      │ HIGH     │
│ 羅東鄉            │ 宜蘭縣    │ 14.3     │ 41.7      │ 15.1     │ 0.0321      │ MEDIUM   │
└─────────────────────┴──────────┴──────────┴──────────┴──────────┴─────────────┴───────────┘
"""
    
    print(expected_output)
    
    # 基於現有資料進行模擬分析
    print(f"\n📈 基於現有資料的模擬分析:")
    
    if 'kriging_results' in globals() and 'rf_results' in globals():
        # 建立模擬的鄉鎮統計
        print(f"🎯 模擬區域統計 (基於網格資料):")
        
        # 建立簡單的區域劃分（基於坐標）
        z_kriging = kriging_results['z_kriging']
        z_rf = rf_results['z_rf']
        
        # 基於坐標劃分為幾個區域
        x_coords = grid_info['grid_xx']
        y_coords = grid_info['grid_yy']
        
        # 簡單的四分區分析
        regions = {
            '東北部': (x_coords < np.median(x_coords)) & (y_coords > np.median(y_coords)),
            '東南部': (x_coords < np.median(x_coords)) & (y_coords <= np.median(y_coords)),
            '西北部': (x_coords >= np.median(x_coords)) & (y_coords > np.median(y_coords)),
            '西南部': (x_coords >= np.median(x_coords)) & (y_coords <= np.median(y_coords))
        }
        
        print(f"{'區域':<8} {'Kriging平均':<12} {'Kriging最大':<12} {'RF平均':<12} {'差異':<8} {'建議行動'}")
        print(f"{'-' * 70}")
        
        for region_name, mask in regions.items():
            if np.any(mask):
                kriging_mean = z_kriging[mask].mean()
                kriging_max = z_kriging[mask].max()
                rf_mean = z_rf[mask].mean()
                diff = kriging_mean - rf_mean
                
                # 決策建議
                if kriging_mean > 20:
                    action = "高風險"
                elif kriging_mean > 10:
                    action = "中風險"
                else:
                    action = "低風險"
                
                print(f"{region_name:<8} {kriging_mean:<12.2f} {kriging_max:<12.2f} {rf_mean:<12.2f} {diff:<8.2f} {action}")
        
        print(f"\n💡 這是一個簡化的區域分析，實際應用需要真實的行政區界線")
    
else:
    print(f"✅ 鄉鎮界線載入成功")
    
    # 篩選目標縣市
    target_counties = ['花蓮縣', '宜蘭縣']
    study_towns = towns[towns['COUNTYNAME'].isin(target_counties)].copy()
    
    print(f"📍 篩選目標縣市: {target_counties}")
    print(f"🏘️ 找到 {len(study_towns)} 個鄉鎮")
    
    # 轉換坐標系統
    if study_towns.crs != 'EPSG:3826':
        study_towns = study_towns.to_crs('EPSG:3826')
        print(f"🗺️ 轉換至 EPSG:3826")
    
    # 計算區域統計
    print(f"\n📊 計算區域統計...")
    
    try:
        # Kriging 降雨統計
        kriging_stats = zonal_stats(
            study_towns, 
            'output/kriging_rainfall.tif',
            stats=['mean', 'max', 'count'],
            nodata=-9999
        )
        
        # Kriging 方差統計
        variance_stats = zonal_stats(
            study_towns,
            'output/kriging_variance.tif', 
            stats=['mean'],
            nodata=-9999
        )
        
        # Random Forest 降雨統計
        rf_stats = zonal_stats(
            study_towns,
            'output/rf_rainfall.tif',
            stats=['mean', 'max', 'count'],
            nodata=-9999
        )
        
        # 建立結果表格
        results = []
        
        for i, town in study_towns.iterrows():
            town_name = town.get('TOWNNAME', f'鄉鎮{i}')
            county_name = town.get('COUNTYNAME', '未知')
            
            # 取得統計資料
            kriging_mean = kriging_stats[i]['mean'] if kriging_stats[i] else 0
            kriging_max = kriging_stats[i]['max'] if kriging_stats[i] else 0
            rf_mean = rf_stats[i]['mean'] if rf_stats[i] else 0
            variance_mean = variance_stats[i]['mean'] if variance_stats[i] else 0
            
            # 計算可信度
            if variance_stats and variance_stats[i]:
                if variance_mean < 0.03:
                    confidence = 'HIGH'
                elif variance_mean < 0.06:
                    confidence = 'MEDIUM'
                else:
                    confidence = 'LOW'
            else:
                confidence = 'UNKNOWN'
            
            results.append({
                '鄉鎮': town_name,
                '縣市': county_name,
                'Kriging平均': round(kriging_mean, 2),
                'Kriging最大': round(kriging_max, 2),
                'RF平均': round(rf_mean, 2),
                '平均variance': round(variance_mean, 4),
                '可信度': confidence
            })
        
        # 轉換為 DataFrame
        df_results = pd.DataFrame(results)
        
        # 顯示結果表格
        print(f"\n📋 鄉鎮決策表格:")
        print(df_results.to_string(index=False))
        
        # ─── 決策分析 ───
        print(f"\n⚡ 決策分析:")
        
        # 高風險鄉鎮
        high_risk = df_results[df_results['Kriging平均'] > 20]
        if len(high_risk) > 0:
            print(f"🚨 高風險鄉鎮 (>20 mm/hr): {len(high_risk)} 個")
            for _, town in high_risk.iterrows():
                print(f"   • {town['鄉鎮']} ({town['縣市']}): {town['Kriging平均']:.1f} mm/hr - {town['可信度']} 信賴度")
        
        # 不確定性高的鄉鎮
        low_confidence = df_results[df_results['可信度'] == 'LOW']
        if len(low_confidence) > 0:
            print(f"\n⚠️ 低信賴度鄉鎮: {len(low_confidence)} 個")
            for _, town in low_confidence.iterrows():
                print(f"   • {town['鄉鎮']} ({town['縣市']}): variance {town['平均variance']:.4f}")
        
        # 方法差異大的鄉鎮
        df_results['差異'] = df_results['Kriging平均'] - df_results['RF平均']
        high_disagreement = df_results[abs(df_results['差異']) > 5]
        if len(high_disagreement) > 0:
            print(f"\n🔍 方法差異大鄉鎮 (>5 mm/hr): {len(high_disagreement)} 個")
            for _, town in high_disagreement.iterrows():
                direction = "Kriging較高" if town['差異'] > 0 else "RF較高"
                print(f"   • {town['鄉鎮']} ({town['縣市']}): {direction} {abs(town['差異']):.1f} mm/hr")
        
        # 儲存結果
        df_results.to_csv('output/township_statistics.csv', index=False, encoding='utf-8-sig')
        print(f"\n💾 結果已儲存至 output/township_statistics.csv")
        
        # 建立決策建議
        print(f"\n🎯 決策建議矩陣:")
        print(f"┌─────────────────┬─────────────────┬─────────────────┐")
        print(f"│ 風險等級 \ 信賴度  │     HIGH        │    LOW/MEDIUM   │")
        print(f"├─────────────────┼─────────────────┼─────────────────┤")
        print(f"│ HIGH (>20mm)    │ 立即疏散        │ 部署監測      │")
        print(f"│ MEDIUM (10-20mm)│ 準備預警        │ 持續監測      │")
        print(f"│ LOW (<10mm)     │ 正常監測        │ 正常監測      │")
        print(f"└─────────────────┴─────────────────┴─────────────────┘")
        
    except Exception as e:
        print(f"❌ 區域統計計算失敗: {e}")
        print(f"💡 可能原因: GeoTIFF 檔案不存在或格式問題")
        
        # 顯示預期格式
        print(f"\n📝 預期輸出格式:")
        print(expected_output)

In [ ]:
from rasterstats import zonal_stats

# YOUR CODE HERE:
# 1. Load township boundaries (TGOS shapefile)
# 2. Filter to 花蓮縣 + 宜蘭縣, convert to EPSG:3826
# 3. Run zonal_stats on kriging_rainfall.tif, kriging_variance.tif, rf_rainfall.tif
# 4. Create summary DataFrame with columns:
#    鄉鎮, 縣市, Kriging平均, Kriging最大, RF平均, 平均variance
# 5. Add 可信度 column:
#    HIGH: variance < 33rd percentile
#    MEDIUM: 33rd-66th percentile
#    LOW: > 66th percentile

# Note: If you don't have the township shapefile, skip this cell
# and describe what the expected output would be in the markdown below.

# try:
#     towns = gpd.read_file('path/to/TOWN_MOI.shp')
#     study_towns = towns[towns['COUNTYNAME'].isin(['花蓮縣', '宜蘭縣'])].copy()
#     study_towns = study_towns.to_crs(epsg=3826)
#     ... (compute zonal stats and create summary table)
# except Exception as e:
#     print(f"Township shapefile not found: {e}")

# 思考挑戰 - 為什麼 ML 不能提供 Sigma Map？

# 請回答以下討論問題：

"""
1. 為什麼 Kriging 原生提供方差地圖，但 Random Forest 不會？

答案：Kriging 基於統計理論（variogram），在估計過程中自然產生方差。RF 是機器學習，專注於預測準確性，沒有內在的不確定性量度。

2. 能否近似 RF 的不確定性？有什麼限制？

答案：可以透過 bootstrap 或樹內變異數近似，但限制是：
- 反映模型不穩定性，非真實預測不確定性
- 缺乏統計基礎
- 無法區分測量誤差和模型誤差

3. 在你的區域統計表中，哪些鄉鎮顯示「高降雨 + 低信賴度」？指揮官應該如何應對？

答案：通常是測站稀疏的山區或邊緣地區。指揮官應該：
- 部署臨時監測設備
- 發布預警但說明不確定性
- 準備疏散計劃
- 持續更新預測
"""

**Your Lab 2 reflection here:**

1. Kriging vs ML uncertainty: ...
2. ML uncertainty approximation: ...
3. High rain + low confidence townships: ...

# (Bonus) AI 顧問諮詢

# 請詢 AI 模型專業建議：

"""
問：在花蓮山區，測站密度約 1 站 / 50 km²。我用 Kriging 和 Random Forest 分別做了降雨內插，結果在山區差異很大。Kriging 的 variance 在山區也很高。請問：(1) 我應該信哪個結果？(2) 如何改善山區的預測品質？

AI 回應範例：

(1) 結果選擇：優先考慮 Kriging，因為統計基礎穩健，高 variance 正確指出了不確定性。

(2) 改善策略：
- 短期：地形整合 Kriging，多源資料融合
- 中期：部署臨時監測，利用民眾回報
- 長期：增加山區測站密度，發展專用模型

我的評論：AI 建議專業實用，特別是重視不確性評估和分層次改善策略。
"""

**AI Response:**

(Paste here)

**My Commentary:**

(Write 2-3 sentences on whether you agree with the AI's advice)

# ARIA 演進回顧
# 從離散點到連續面的技術進展

print("🌟 ARIA 系統演進回顧")
print("📈 每週的技術突破與能力提升")

# ─── 系統演進表格 ───
evolution_data = [
    {
        'Version': 'v1.0',
        'Week': 'W3',
        'Capability': 'River buffer + shelters',
        'Key Question': 'How close to the river?',
        'Technical Focus': 'GIS 緩衝區分析',
        'Achievement': '基礎空間關聯分析'
    },
    {
        'Version': 'v2.0', 
        'Week': 'W4',
        'Capability': '+ DEM terrain analysis',
        'Key Question': 'How steep is the terrain?',
        'Technical Focus': '地形因子整合',
        'Achievement': '多維度風險評估'
    },
    {
        'Version': 'v3.0',
        'Week': 'W5', 
        'Capability': '+ Real-time rainfall stations',
        'Key Question': 'How much rain RIGHT NOW?',
        'Technical Focus': '即時資料整合',
        'Achievement': '動態風險監測系統'
    },
    {
        'Version': 'v3.5',
        'Week': 'W6',
        'Capability': '+ Kriging & ML interpolation',
        'Key Question': 'What about areas with NO station? How confident are we?',
        'Technical Focus': '空間內插與不確定性',
        'Achievement': '連續表面預測與信賴度評估'
    },
    {
        'Version': 'v4.0',
        'Week': 'Final Project',
        'Capability': 'Your extension!',
        'Key Question': 'Your question!',
        'Technical Focus': '創新擴展',
        'Achievement': '個人化專案貢獻'
    }
]

print(f"\n📊 ARIA 系統演進歷程:")
print(f"{'版本':<8} {'週次':<6} {'核心能力':<25} {'關鍵問題':<35} {'技術重點':<20}")
print(f"{'-' * 110}")

for version in evolution_data:
    print(f"{version['Version']:<8} {version['Week']:<6} {version['Capability']:<25} {version['Key Question']:<35} {version['Technical Focus']:<20}")

print(f"{'-' * 110}")

# ─── 技術能力進展分析 ───
print(f"\n🔍 技術能力進展分析:")

print(f"\n🌊 W3: 河流緩衝區分析")
print(f"   📏 核心技術: 歐幾里得距離緩衝區")
print(f"   🎯 解決問題: 避難所與河流的空間關係")
print(f"   💡 技術突破: 從點資料到空間影響範圍")
print(f"   ⚡ 限制: 僅考慮距離，忽略地形因素")

print(f"\n🏔️ W4: 地形分析整合")
print(f"   📏 核心技術: DEM 高程、坡度分析")
print(f"   🎯 解決問題: 地形複雜度對避難所影響")
print(f"   💡 技術突破: 多維度風險評估框架")
print(f"   ⚡ 限制: 靜態地形，缺乏動態因素")

print(f"\n🌧️ W5: 即時降雨整合")
print(f"   📏 核心技術: API 資料獲取、動態風險分級")
print(f"   🎯 解決問題: 即時氣象條件下的風險評估")
print(f"   💡 技術突破: 動態監測系統架構")
print(f"   ⚡ 限制: 僅限測站位置，無法預測無測站區域")

print(f"\n🗺️ W6: 空間內插與不確定性")
print(f"   📏 核心技術: Kriging、Random Forest、Sigma Map")
print(f"   🎯 解決問題: 無測站區域的降雨預測與信賴度")
print(f"   💡 技術突破: 從離散點到連續面的完整轉換")
print(f"   ⚡ 限制: 需要更多驗證資料")

# ─── 方法論演進 ───
print(f"\n🔬 方法論演進:")

methodology_evolution = [
    ("W3", "簡單緩衝區", "距離計算", "GIS 基礎操作"),
    ("W4", "多因子加權", "地形分析", "綜合評估"),
    ("W5", "動態分級", "即時資料", "時間維度整合"),
    ("W6", "統計內插", "空間統計", "科學預測")
]

print(f"{'週次':<6} {'方法':<12} {'技術':<12} {'層次'}")
print(f"{'-' * 45}")
for week, method, tech, level in methodology_evolution:
    print(f"{week:<6} {method:<12} {tech:<12} {level}")

# ─── 決策支援能力提升 ───
print(f"\n⚡ 決策支援能力提升:")

decision_support = [
    ("W3", "避難所選址", "靜態評估", "規劃階段"),
    ("W4", "風險分級", "綜合評估", "預防措施"),
    ("W5", "即時預警", "動態監測", "應變決策"),
    ("W6", "不確定性評估", "科學決策", "精準防災")
]

print(f"{'版本':<6} {'決策類型':<12} {'支援方式':<12} {'應用階段'}")
print(f"{'-' * 50}")
for version, decision_type, support, stage in decision_support:
    print(f"{version:<6} {decision_type:<12} {support:<12} {stage}")

# ─── 技術挑戰與解決方案 ───
print(f"\n🛠️ 技術挑戰與解決方案:")

challenges_solutions = [
    {
        '挑戰': '資料稀疏性',
        'W3解決': 'N/A',
        'W4解決': 'N/A', 
        'W5解決': 'API fallback 機制',
        'W6解決': 'Kriging 內插 + Sigma Map'
    },
    {
        '挑戰': '地形複雜度',
        'W3解決': '忽略地形',
        'W4解決': 'DEM 整合',
        'W5解決': '維持 W4 功能',
        'W6解決': '地形相關 variogram'
    },
    {
        '挑戰': '即時性要求',
        'W3解決': '靜態分析',
        'W4解決': '靜態分析',
        'W5解決': 'API 即時更新',
        'W6解決': '高效內插演算法'
    },
    {
        '挑戰': '決策信心度',
        'W3解決': '二元判斷',
        'W4解決': '分級評估',
        'W5解決': '動態分級',
        'W6解決': '不確定性量化'
    }
]

print(f"{'挑戰':<12} {'W3':<12} {'W4':<12} {'W5':<12} {'W6':<12}")
print(f"{'-' * 65}")
for item in challenges_solutions:
    print(f"{item['挑戰']:<12} {item['W3解決']:<12} {item['W4解決']:<12} {item['W5解決']:<12} {item['W6解決']:<12}")

# ─── 未來發展方向 ───
print(f"\n🚀 未來發展方向 (v4.0+):")

future_directions = [
    "🌐 多源資料融合 (衛星、雷達、物聯網)",
    "🤖 深度學習時空預測模型",
    "📱 行動化決策支援系統",
    "🔗 跨區域協防網絡",
    "📊 預測不確定性傳播分析",
    "🌍 氣候變遷適應性調整",
    "⚡ 邊緣計算即時處理",
    "🎯 個人化風險評估"
]

print(f"潛在創新方向:")
for i, direction in enumerate(future_directions, 1):
    print(f"{i:2d}. {direction}")

# ─── 學習成果總結 ───
print(f"\n🎓 學習成果總結:")
print(f"📚 技術技能:")
print(f"   ✅ GIS 空間分析與緩衝區操作")
print(f"   ✅ DEM 地形分析與多因子整合") 
print(f"   ✅ API 資料獲取與動態系統設計")
print(f"   ✅ 空間統計內插與機器學習預測")
print(f"   ✅ 不確定性評估與決策支援")

print(f"\n🔬 科學思維:")
print(f"   ✅ 從離散到連續的空間思維轉換")
print(f"   ✅ 多尺度分析能力")
print(f"   ✅ 不確定性量化意識")
print(f"   ✅ 實務導向的問題解決")

print(f"\n💡 工程實踐:")
print(f"   ✅ 系統架構設計與模組化開發")
print(f"   ✅ 錯誤處理與防呆機制")
print(f"   ✅ 效能優化與計算效率")
print(f"   ✅ 專業文檔與結果呈現")

print(f"\n🌟 核心價值:")
print(f"   📊 '內插不僅是填補空間；它是預測傳感器無法觸及之處的風險'")
print(f"   🛡️ '不確定性不是弱點；它是謹慎決策的指南'")
print(f"   🎯 '技術進步的目標是讓決策更明智，而不只是更快速'")

print(f"\n✨ ARIA v3.5 - 空間預測對決完成")
print(f"🚀 準備迎接 v4.0 的挑戰！")